# Phase-Aware Lap Bar and Bottle Position Measurement

This notebook keeps the same phase-aware foundation as the existing repo work, but it uses a **reference image** instead of a reference clip.

The reference image is only used to align phase labels across videos so that `phase 0` means the same visual state everywhere. The measurement work then stays frame-based:

1. Load a reference image and choose the ROI.
2. Detect repeating phases independently for each video.
3. Rotate each video's phase grid so the phase that best matches the reference image becomes aligned `phase 0`.
4. Export Phase 2 frames from the reference clip.
5. Apply the Pitwall config masks and review them overlaid on the Phase 2 cycles.
6. Save **per-frame** lap bar and bottle measurements to CSV.
7. Start with scatter plots and only aggregate later if the data justifies it.


In [1]:
from pathlib import Path
import json
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import pandas as pd

%matplotlib inline

# Work out where the notebook is being run from.
current_dir = Path.cwd()
if current_dir.name == "measurements" and (current_dir / "phase_aware_position_output").exists():
    notebook_dir = current_dir
    repo_root = current_dir.parent
elif (current_dir / "measurements").exists():
    repo_root = current_dir
    notebook_dir = current_dir / "measurements"
else:
    raise RuntimeError(
        "Open this notebook from the repo root or from the measurements folder. "
        f"Current directory was: {current_dir}"
    )

stoppage_detection_dir = repo_root / "stoppage_detection"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

# Existing repo functions used for phase detection and phase-grid construction.
from VideoModule.io.read import read_video, build_roi_phase_grid
from VideoModule.preprocessing import resample_segments_to_phases
from VideoModule.image_similarity import select_roi_on_frames
from VideoModule.image_similarity.roi import crop_to_roi, load_roi, save_roi
from VideoModule.anomaly_detection.cohort_anomaly_detection import compute_ncc_opencv
from VideoModule.plotting import plot_dynamic_phase_overview, plot_phase_grid
from VideoModule.phase_detection import run_phase_awareness
from VideoModule.data_classes.geometry import Roi
from VideoModule.data_classes.phase.cfg import PhaseDetectionConfig

## 1. Settings and Core Functions

**What this section does**

Set the paths, phase-aware config, and the small helper functions that drive the rest of the notebook.

**Existing repo functions used here**

- `PhaseDetectionConfig`
- `read_video()`
- `run_phase_awareness()`
- `detect_dynamic_phases()` through `run_phase_awareness()`
- `resample_segments_to_phases()`
- `build_roi_phase_grid()`
- `compute_ncc_opencv()`


In [2]:
# ------------------------------
# User settings
# ------------------------------

# Reference assets live here.
reference_dir = notebook_dir / "reference_clip"

# This image defines what aligned phase 0 should look like.
reference_image_path = reference_dir / "vlcsnap-2026-07-02-22h22m16s628.png"

# These are the videos that will be phase-detected and exported for Pitwall.
video_dir = stoppage_detection_dir / "stop_clips"

# Everything this notebook writes will go here.
output_dir = notebook_dir / "phase_aware_position_output"
output_config_dir = output_dir / "config"
output_manifest_dir = output_dir / "manifests"
output_measurements_dir = output_dir / "measurements"
output_frames_dir = output_dir / "frames"
output_masks_dir = output_dir / "masks"
output_overlays_dir = output_dir / "overlays"
output_batch_dir = output_dir / "batch"

exported_frames_dir = output_frames_dir / "phase_exports"
pitwall_mask_output_dir = output_masks_dir / "pitwall"
roi_json_path = output_config_dir / "measurement_roi.json"
measurement_template_path = output_manifest_dir / "pitwall_measurement_template.csv"
measurements_output_path = output_measurements_dir / "pitwall_position_measurements.csv"

# Phase 2 is the only phase used from this point in the workflow.
phase_to_review = 3
phases_of_interest = [phase_to_review]

# Pitwall config exports used to build lapbar, bottle, and datum masks.
pitwall_config_dir = notebook_dir / "pitwall_config"
lapbar_config_path = pitwall_config_dir / "lapbar_v3.json"
bottle_config_path = pitwall_config_dir / "bottles_v3.json"
datum_config_path = pitwall_config_dir / "datum.json"

# Toggle the measurement ROI crop on or off.
# This ROI is for measurement/alignment scoring only. Phase detection stays full-frame by default.
use_measurement_roi = True

# Optional separate ROI for phase detection. Keep None unless this ROI contains the repeating motion.
phase_detection_roi = None

# Choose which clips Chapter 4 exports from: "reference_clip" or "stop_clips".
export_clip_source = "reference_clip"

# Limit export volume so the first pass stays manageable.
# Use None to export every matching phase instance.
export_samples_per_phase = None
max_videos = None

# Keep the phase settings close to the master phase-aware notebook.
# Video loading is notebook-level because PhaseDetectionConfig only owns phase detection.
phase_max_frames = None
phase_load_resize = None  # None keeps native video resolution.
phase_ncc_resize_dim = (256, 256)  # Used only for the reference/key-frame NCC scoring below.

cfg = PhaseDetectionConfig(
    energy_method="ncc",
    use_fcwt=True,
    min_hz=0.2,
    max_hz=5.0,
    min_region_seconds=5.0,
    min_freq_change_hz=2,
    num_phases=10,
    phase_binning="linear_time",
    phase_roi=phase_detection_roi,
)

output_dir.mkdir(parents=True, exist_ok=True)
for output_subdir in [
    output_config_dir,
    output_manifest_dir,
    output_measurements_dir,
    output_frames_dir,
    output_masks_dir,
    output_overlays_dir,
    output_batch_dir,
    exported_frames_dir,
    pitwall_mask_output_dir,
]:
    output_subdir.mkdir(parents=True, exist_ok=True)


def list_image_paths(folder_path: Path, max_items: int | None = None) -> list[Path]:
    """Find candidate image files in a folder."""
    allowed_suffixes = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    image_paths = sorted(
        path for path in folder_path.iterdir() if path.is_file() and path.suffix.lower() in allowed_suffixes
    )
    if max_items is not None:
        image_paths = image_paths[:max_items]
    return image_paths


def resolve_reference_image_path(reference_dir: Path, configured_path) -> Path:
    """Pick the configured reference image, or auto-detect one from reference_clip."""
    if configured_path is not None:
        configured_path = Path(configured_path)
        if configured_path.exists():
            return configured_path
        raise FileNotFoundError(f"Configured reference image was not found: {configured_path}")

    image_paths = list_image_paths(reference_dir, max_items=1)
    if not image_paths:
        raise FileNotFoundError(
            f"No reference image was found in: {reference_dir}. Add one image or set reference_image_path explicitly."
        )
    return image_paths[0]


def load_reference_image(image_path: Path) -> np.ndarray:
    """Load the reference image that defines aligned phase 0."""
    reference_image = cv2.imread(str(image_path))
    if reference_image is None:
        raise FileNotFoundError(
            f"Could not read reference image: {image_path}. Update reference_image_path first."
        )
    return reference_image


def list_video_paths(folder_path: Path, max_items: int | None = None) -> list[Path]:
    """Find candidate video files in a folder."""
    allowed_suffixes = {".ts", ".mp4", ".avi", ".mov", ".mkv"}
    video_paths = sorted(
        path for path in folder_path.iterdir() if path.is_file() and path.suffix.lower() in allowed_suffixes
    )
    if max_items is not None:
        video_paths = video_paths[:max_items]
    if not video_paths:
        raise ValueError(f"No videos were found in: {folder_path}")
    return video_paths


def prepare_ncc_image(frame: np.ndarray, roi, resize_dim: tuple[int, int]) -> np.ndarray:
    """Crop to the ROI, convert to grayscale, and resize for NCC scoring."""
    cropped_frame = crop_to_roi(frame, roi)
    grayscale_frame = cv2.cvtColor(cropped_frame, cv2.COLOR_BGR2GRAY)
    return cv2.resize(grayscale_frame, resize_dim, interpolation=cv2.INTER_AREA)


def detect_video_phases(video_path: Path, cfg: PhaseDetectionConfig) -> dict:
    """Run the repo phase-awareness component on one video."""
    # Load frames once at the notebook-selected resolution.
    # phase_load_resize=None keeps the native video resolution.
    frames, fps = read_video(video_path, max_frames=phase_max_frames, resize=phase_load_resize)

    # Use the current VideoModule phase-awareness component so phase_roi is applied consistently.
    phase_awareness = run_phase_awareness(frames=frames, fps=fps, config=cfg)
    phase_result = phase_awareness.dyn

    if len(phase_result.cycles) < 1:
        raise ValueError(f"No cycles were found in: {video_path.name}")

    return {
        "video_path": video_path,
        "frames": frames,
        "fps": fps,
        "energy_signal": phase_awareness.energy_signal,
        "energy_method_used": cfg.energy_method,
        "phase_result": phase_result,
        "phase_awareness": phase_awareness,
        "preview_phase_grid": phase_awareness.phase_grid,
    }


def align_video_phases_to_reference_image(
    reference_image: np.ndarray,
    video_path: Path,
    cycles: list[tuple[int, int]],
    cfg: PhaseDetectionConfig,
    roi,
) -> dict:
    """Score phases in the ROI, but keep exported/reviewed frames full-frame."""
    # Build a cropped grid only for NCC scoring against the reference image.
    scoring_phase_grid = build_roi_phase_grid(
        video_path=video_path,
        cycles=cycles,
        num_phases=cfg.num_phases,
        roi=roi,
        resize=None,
    )

    if not scoring_phase_grid:
        raise ValueError(f"Could not build an ROI scoring phase grid for: {video_path.name}")

    # Build a matching full-frame grid for plotting, exporting, masks, and measurements.
    full_frame_phase_grid = build_roi_phase_grid(
        video_path=video_path,
        cycles=cycles,
        num_phases=cfg.num_phases,
        roi=None,
        resize=None,
    )

    if not full_frame_phase_grid:
        raise ValueError(f"Could not build a full-frame phase grid for: {video_path.name}")

    paired_cycle_count = min(len(scoring_phase_grid), len(full_frame_phase_grid))
    scoring_phase_grid = scoring_phase_grid[:paired_cycle_count]
    full_frame_phase_grid = full_frame_phase_grid[:paired_cycle_count]

    reference_roi_image = prepare_ncc_image(reference_image, roi, phase_ncc_resize_dim)

    phase_scores = []
    for original_phase_index in range(cfg.num_phases):
        cycle_scores = []
        for scoring_cycle_frames in scoring_phase_grid:
            candidate_frame = prepare_ncc_image(scoring_cycle_frames[original_phase_index], None, phase_ncc_resize_dim)
            cycle_scores.append(compute_ncc_opencv(candidate_frame, reference_roi_image))
        phase_scores.append(float(np.median(cycle_scores)))

    best_original_phase_index = int(np.argmax(phase_scores))

    aligned_phase_grid = []
    frame_records = []
    for cycle_index, full_cycle_frames in enumerate(full_frame_phase_grid):
        aligned_cycle_frames = full_cycle_frames[best_original_phase_index:] + full_cycle_frames[:best_original_phase_index]
        aligned_phase_grid.append(aligned_cycle_frames)

        for aligned_phase_index, frame in enumerate(aligned_cycle_frames):
            original_phase_index = (aligned_phase_index + best_original_phase_index) % cfg.num_phases
            frame_records.append(
                {
                    "video_name": video_path.name,
                    "cycle_index": cycle_index,
                    "aligned_phase_index": aligned_phase_index,
                    "original_phase_index": original_phase_index,
                    "phase_zero_source_phase_index": best_original_phase_index,
                    "phase_zero_match_score": phase_scores[best_original_phase_index],
                    "frame": frame,
                }
            )

    phase_score_df = pd.DataFrame(
        {
            "original_phase_index": list(range(cfg.num_phases)),
            "median_ncc_to_reference_image": phase_scores,
        }
    )

    return {
        "roi_phase_grid": scoring_phase_grid,
        "full_frame_phase_grid": full_frame_phase_grid,
        "aligned_phase_grid": aligned_phase_grid,
        "best_original_phase_index": best_original_phase_index,
        "phase_score_df": phase_score_df,
        "frame_records": frame_records,
    }


def select_evenly_spaced_items(items: list[dict], max_items: int) -> list[dict]:
    """Keep a stable spread of rows instead of only the earliest rows."""
    if len(items) <= max_items:
        return items
    selected_positions = np.linspace(0, len(items) - 1, max_items).round().astype(int)
    return [items[index] for index in selected_positions]


def export_phase_frames(frame_records: list[dict], export_dir: Path, phases_of_interest: list[int], samples_per_phase: int | None) -> pd.DataFrame:
    """Export full-frame frames for the selected aligned phases."""
    exported_rows = []

    for aligned_phase_index in phases_of_interest:
        matching_records = [
            record for record in frame_records if record["aligned_phase_index"] == aligned_phase_index
        ]
        matching_records = sorted(
            matching_records,
            key=lambda record: (record["video_name"], record["cycle_index"], record["original_phase_index"]),
        )
        if samples_per_phase is None:
            selected_records = matching_records
        else:
            selected_records = select_evenly_spaced_items(matching_records, samples_per_phase)

        for export_index, record in enumerate(selected_records):
            video_stem = Path(record["video_name"]).stem
            file_name = (
                f"phase_{aligned_phase_index:02d}__{video_stem}"
                f"__cycle_{record['cycle_index']:04d}"
                f"__original_phase_{record['original_phase_index']:02d}"
                f"__sample_{export_index:03d}.png"
            )
            export_path = export_dir / file_name
            cv2.imwrite(str(export_path), record["frame"])

            exported_rows.append(
                {
                    "video_name": record["video_name"],
                    "cycle_index": record["cycle_index"],
                    "aligned_phase_index": aligned_phase_index,
                    "original_phase_index": record["original_phase_index"],
                    "phase_zero_source_phase_index": record["phase_zero_source_phase_index"],
                    "phase_zero_match_score": record["phase_zero_match_score"],
                    "keyframe_ncc_score": record.get("keyframe_ncc_score", record["phase_zero_match_score"]),
                    "keyframe_ncc_margin": record.get("keyframe_ncc_margin", np.nan),
                    "keyframe_frame_index": record.get("keyframe_frame_index", np.nan),
                    "keyframe_cycle_start": record.get("keyframe_cycle_start", np.nan),
                    "keyframe_cycle_end": record.get("keyframe_cycle_end", np.nan),
                    "keyframe_candidate_count": record.get("keyframe_candidate_count", np.nan),
                    "phase_selection_method": record.get("phase_selection_method", "phase_zero_alignment"),
                    "export_path": str(export_path),
                }
            )

    return pd.DataFrame(exported_rows)


def build_phase_frame_manifest(exported_manifest_df: pd.DataFrame) -> pd.DataFrame:
    """Add the empty columns that Pitwall output will populate later."""
    manifest_df = exported_manifest_df.copy()
    manifest_df["lapbar_mask_path"] = ""
    manifest_df["bottle_mask_path"] = ""
    manifest_df["datum_mask_path"] = ""
    manifest_df["lapbar_x_from_datum"] = np.nan
    manifest_df["lapbar_y_from_datum"] = np.nan
    manifest_df["bottle_x_from_datum"] = np.nan
    manifest_df["bottle_y_from_datum"] = np.nan
    manifest_df["datum_x_from_frame"] = np.nan
    manifest_df["datum_y_from_frame"] = np.nan
    manifest_df["bottle_roi_count"] = np.nan
    manifest_df["bottle_measurement_valid"] = False
    manifest_df["bottle_middle_roi_full_size"] = False
    manifest_df["bottle_minus_datum_x"] = np.nan
    manifest_df["bottle_minus_datum_y"] = np.nan
    manifest_df["bottle_from_datum_euclidean"] = np.nan
    manifest_df["lapbar_minus_datum_x"] = np.nan
    manifest_df["lapbar_minus_datum_y"] = np.nan
    manifest_df["lapbar_from_datum_euclidean"] = np.nan
    return manifest_df


def load_pitwall_config(config_path: Path) -> dict:
    """Load one Pitwall JSON config file."""
    # Read the JSON export from the Pitwall config folder.
    with open(config_path, "r", encoding="utf-8") as config_file:
        pitwall_config = json.load(config_file)

    # Return the raw config so each processing step can read its own settings.
    return pitwall_config


def ensure_odd_kernel_size(raw_kernel_size: int | float, minimum: int = 1) -> int:
    """Convert UI kernel values into OpenCV-safe odd kernel sizes."""
    # Convert the UI value to a positive integer.
    kernel_size = max(int(round(raw_kernel_size)), minimum)

    # OpenCV median and Gaussian filters need odd kernel sizes.
    if kernel_size % 2 == 0:
        kernel_size += 1

    # Return a value that can be passed directly to OpenCV.
    return kernel_size


def clip_rectangle_to_frame(roi_record: dict, frame_shape: tuple[int, ...]) -> tuple[int, int, int, int]:
    """Clamp a Pitwall rectangle ROI so it fits inside the frame."""
    # Read the frame dimensions in image coordinates.
    frame_height, frame_width = frame_shape[:2]

    # Read and clamp the rectangle start point.
    x1 = max(0, int(roi_record.get("x", 0)))
    y1 = max(0, int(roi_record.get("y", 0)))

    # Read and clamp the rectangle end point.
    x2 = min(frame_width, x1 + max(0, int(roi_record.get("width", 0))))
    y2 = min(frame_height, y1 + max(0, int(roi_record.get("height", 0))))

    # Return the rectangle in slice-friendly coordinates.
    return x1, y1, x2, y2


def create_roi_mask(frame_shape: tuple[int, ...], pitwall_config: dict) -> np.ndarray:
    """Create the enabled include/exclude ROI mask from a Pitwall config."""
    # Start with a black mask until an include ROI says which area matters.
    frame_height, frame_width = frame_shape[:2]
    roi_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)

    # If ROI handling is disabled, treat the whole frame as valid.
    roi_settings = pitwall_config.get("roi", {})
    if not roi_settings.get("enabled", False):
        roi_mask[:, :] = 255
        return roi_mask

    # Keep the full frame when the config has no active ROI records.
    active_rois = roi_settings.get("active_rois", [])
    enabled_rois = [roi_record for roi_record in active_rois if roi_record.get("enabled", True)]
    if not enabled_rois:
        roi_mask[:, :] = 255
        return roi_mask

    # Fill include rectangles first.
    for roi_record in enabled_rois:
        if roi_record.get("mode", "include") != "include":
            continue
        x1, y1, x2, y2 = clip_rectangle_to_frame(roi_record, frame_shape)
        roi_mask[y1:y2, x1:x2] = 255

    # If all active ROIs are exclusions, start from the full frame.
    if not roi_mask.any():
        roi_mask[:, :] = 255

    # Clear exclude rectangles after includes have been applied.
    for roi_record in enabled_rois:
        if roi_record.get("mode", "include") != "exclude":
            continue
        x1, y1, x2, y2 = clip_rectangle_to_frame(roi_record, frame_shape)
        roi_mask[y1:y2, x1:x2] = 0

    # Return a binary uint8 mask where white means the ROI is active.
    return roi_mask


def create_hue_mask(frame: np.ndarray, hue_settings: dict) -> np.ndarray:
    """Create a hue/saturation/value mask from Pitwall hue settings."""
    # Convert the BGR frame into OpenCV HSV coordinates.
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Prefer the explicit min/max values saved by the Pitwall UI.
    hue_min = int(hue_settings.get("hue_min", hue_settings.get("target_hue", 0)))
    hue_max = int(hue_settings.get("hue_max", hue_settings.get("target_hue", 180)))
    saturation_min = int(hue_settings.get("saturation_min", 0))
    saturation_max = int(hue_settings.get("saturation_max", 255))
    value_min = int(hue_settings.get("value_min", 0))
    value_max = int(hue_settings.get("value_max", 255))

    # Clamp values to the OpenCV HSV ranges.
    hue_min = max(0, min(179, hue_min))
    hue_max = max(0, min(179, hue_max))
    saturation_min = max(0, min(255, saturation_min))
    saturation_max = max(0, min(255, saturation_max))
    value_min = max(0, min(255, value_min))
    value_max = max(0, min(255, value_max))

    # Build a normal hue range mask when the range does not wrap around red.
    if hue_min <= hue_max:
        lower_bound = np.array([hue_min, saturation_min, value_min], dtype=np.uint8)
        upper_bound = np.array([hue_max, saturation_max, value_max], dtype=np.uint8)
        return cv2.inRange(hsv_frame, lower_bound, upper_bound)

    # Build two masks when the hue range wraps around the 0/179 boundary.
    lower_mask = cv2.inRange(
        hsv_frame,
        np.array([hue_min, saturation_min, value_min], dtype=np.uint8),
        np.array([179, saturation_max, value_max], dtype=np.uint8),
    )
    upper_mask = cv2.inRange(
        hsv_frame,
        np.array([0, saturation_min, value_min], dtype=np.uint8),
        np.array([hue_max, saturation_max, value_max], dtype=np.uint8),
    )
    return cv2.bitwise_or(lower_mask, upper_mask)


def apply_edge_detection_step(frame: np.ndarray, current_mask: np.ndarray, edge_settings: dict) -> np.ndarray:
    """Apply an optional Canny edge step and keep edges inside the current mask."""
    # Use the current binary mask or the source image as the Canny input.
    if edge_settings.get("source", "image") == "mask":
        edge_source = current_mask
    else:
        edge_source = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Convert the UI settings into OpenCV Canny parameters.
    threshold1 = int(edge_settings.get("threshold1", 50))
    threshold2 = int(edge_settings.get("threshold2", 150))
    aperture_size = ensure_odd_kernel_size(edge_settings.get("aperture_size", 3), minimum=3)
    aperture_size = min(aperture_size, 7)

    # Detect edges, then keep only edges that are inside the previous mask.
    edge_mask = cv2.Canny(edge_source, threshold1, threshold2, apertureSize=aperture_size)
    return cv2.bitwise_and(current_mask, edge_mask)


def apply_morphology_step(current_mask: np.ndarray, morphology_settings: dict) -> np.ndarray:
    """Apply the optional Pitwall morphology step to a binary mask."""
    # Build a small structuring element from the UI settings.
    kernel_size = ensure_odd_kernel_size(morphology_settings.get("kernel_size", 3), minimum=1)
    iterations = max(1, int(morphology_settings.get("iterations", 1)))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

    # Map the UI operation names onto simple OpenCV operations.
    operation = morphology_settings.get("operation", "open")
    if operation == "erode":
        processed_mask = cv2.erode(current_mask, kernel, iterations=iterations)
    elif operation == "dilate":
        processed_mask = cv2.dilate(current_mask, kernel, iterations=iterations)
    elif operation in {"close", "closing"}:
        processed_mask = cv2.morphologyEx(current_mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    else:
        processed_mask = cv2.morphologyEx(current_mask, cv2.MORPH_OPEN, kernel, iterations=iterations)

    # Threshold the result back to a clean binary mask.
    _, binary_mask = cv2.threshold(processed_mask, 0, 255, cv2.THRESH_BINARY)
    return binary_mask


def apply_denoising_step(current_mask: np.ndarray, denoising_settings: dict) -> np.ndarray:
    """Apply optional denoising to a binary mask."""
    # Read which filter the Pitwall UI saved.
    filter_type = denoising_settings.get("filter_type", "median")
    kernel_size = ensure_odd_kernel_size(denoising_settings.get("kernel_size", 5), minimum=3)

    # Median is useful for speckled binary masks.
    if filter_type == "median":
        processed_mask = cv2.medianBlur(current_mask, kernel_size)

    # Gaussian is useful when a later threshold should smooth a jagged mask.
    elif filter_type == "gaussian":
        sigma = float(denoising_settings.get("sigma", 0.0))
        processed_mask = cv2.GaussianBlur(current_mask, (kernel_size, kernel_size), sigma)

    # Bilateral is supported for completeness, but it is still thresholded afterwards.
    else:
        sigma_color = float(denoising_settings.get("sigma_color", 75.0))
        sigma_space = float(denoising_settings.get("sigma_space", 75.0))
        processed_mask = cv2.bilateralFilter(current_mask, kernel_size, sigma_color, sigma_space)

    # Convert any soft filter output back into a binary mask.
    _, binary_mask = cv2.threshold(processed_mask, 0, 255, cv2.THRESH_BINARY)
    return binary_mask


def contour_passes_filter(contour: np.ndarray, contour_settings: dict) -> bool:
    """Check one contour against the Pitwall contour filter settings."""
    # Measure the basic contour geometry.
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    x, y, width, height = cv2.boundingRect(contour)

    # Reject contours outside the saved area and perimeter ranges.
    if area < float(contour_settings.get("min_area", 0)):
        return False
    if area > float(contour_settings.get("max_area", np.inf)):
        return False
    if perimeter < float(contour_settings.get("min_perimeter", 0)):
        return False
    if perimeter > float(contour_settings.get("max_perimeter", np.inf)):
        return False

    # Reject contours outside the saved aspect-ratio range.
    aspect_ratio = float(width) / float(height) if height else 0.0
    if aspect_ratio < float(contour_settings.get("min_aspect_ratio", 0.0)):
        return False
    if aspect_ratio > float(contour_settings.get("max_aspect_ratio", np.inf)):
        return False

    # Compute circularity only when the contour has a perimeter.
    circularity = 0.0
    if perimeter > 0:
        circularity = (4.0 * np.pi * area) / (perimeter ** 2)
    if circularity < float(contour_settings.get("min_circularity", 0.0)):
        return False
    if circularity > float(contour_settings.get("max_circularity", np.inf)):
        return False

    # Use hull solidity as the practical convexity check.
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    convexity = float(area) / float(hull_area) if hull_area else 0.0
    if convexity < float(contour_settings.get("min_convexity", 0.0)):
        return False
    if convexity > float(contour_settings.get("max_convexity", np.inf)):
        return False

    # Keep contours that pass every configured filter.
    return True


def apply_contour_filtering_step(current_mask: np.ndarray, contour_settings: dict) -> np.ndarray:
    """Filter a binary mask down to contours that match the Pitwall settings."""
    # Find external contours in the current binary mask.
    contours, _ = cv2.findContours(current_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Draw only the contours that pass the saved filter thresholds.
    filtered_mask = np.zeros_like(current_mask)
    for contour in contours:
        if contour_passes_filter(contour, contour_settings):
            cv2.drawContours(filtered_mask, [contour], -1, 255, thickness=cv2.FILLED)

    # Return the filtered binary mask.
    return filtered_mask


def apply_shape_extraction_step(current_mask: np.ndarray, shape_settings: dict) -> np.ndarray:
    """Apply optional line extraction when the Pitwall config asks for it."""
    # Only hough_lines is represented in the current Pitwall config files.
    if shape_settings.get("shape_type") != "hough_lines":
        return current_mask

    # Run a probabilistic Hough transform on the current mask.
    threshold = int(shape_settings.get("hough_threshold", 50))
    min_line_length = int(shape_settings.get("hough_min_line_length", 50))
    max_line_gap = int(shape_settings.get("hough_max_line_gap", 10))
    lines = cv2.HoughLinesP(
        current_mask,
        rho=1,
        theta=np.pi / 180,
        threshold=threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap,
    )

    # Keep the region mask unchanged if no lines are found.
    if lines is None:
        return current_mask

    # Draw the detected lines into a separate mask.
    line_mask = np.zeros_like(current_mask)
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(line_mask, (x1, y1), (x2, y2), 255, thickness=3)

    # Return the extracted line representation for review and measurement.
    return line_mask


def build_pitwall_mask(frame: np.ndarray, pitwall_config: dict) -> np.ndarray:
    """Apply a Pitwall config to one BGR frame and return a binary mask."""
    # Start from the configured ROI so every later step stays spatially constrained.
    roi_mask = create_roi_mask(frame.shape, pitwall_config)
    current_mask = roi_mask.copy()

    # Apply processors in the same order recorded by the Pitwall UI.
    processor_order = pitwall_config.get("app", {}).get("mask_processor_order", [])
    for processor_name in processor_order:
        if processor_name == "edge_detection":
            edge_settings = pitwall_config.get("edge_detection", {})
            if edge_settings.get("enabled", False):
                current_mask = apply_edge_detection_step(frame, current_mask, edge_settings)

        elif processor_name == "hue_detection":
            hue_settings = pitwall_config.get("hue_detection", {})
            if hue_settings.get("enabled", False):
                hue_mask = create_hue_mask(frame, hue_settings)
                current_mask = cv2.bitwise_and(current_mask, hue_mask)

        elif processor_name == "contour_filtering":
            contour_settings = pitwall_config.get("contour_filtering", {})
            if contour_settings.get("enabled", False):
                current_mask = apply_contour_filtering_step(current_mask, contour_settings)

        elif processor_name == "morphology":
            morphology_settings = pitwall_config.get("morphology", {})
            if morphology_settings.get("enabled", False):
                current_mask = apply_morphology_step(current_mask, morphology_settings)

        elif processor_name == "denoising":
            denoising_settings = pitwall_config.get("denoising", {})
            if denoising_settings.get("enabled", False):
                current_mask = apply_denoising_step(current_mask, denoising_settings)

        elif processor_name == "shape_extraction":
            shape_settings = pitwall_config.get("shape_extraction", {})
            if shape_settings.get("enabled", False):
                current_mask = apply_shape_extraction_step(current_mask, shape_settings)

        # Keep every processor restricted to the configured ROI.
        current_mask = cv2.bitwise_and(current_mask, roi_mask)

    # Return a clean binary mask.
    _, binary_mask = cv2.threshold(current_mask, 0, 255, cv2.THRESH_BINARY)
    return binary_mask


def overlay_mask_on_frame(frame: np.ndarray, mask: np.ndarray, colour_bgr: tuple[int, int, int], alpha: float) -> np.ndarray:
    """Blend one binary mask onto a BGR frame."""
    # Make a copy so the source frame is never modified in place.
    overlay_frame = frame.copy()

    # Skip gracefully when a mask was not produced for this frame.
    if mask is None:
        return overlay_frame

    # Keep the overlay on the full frame, but only when the mask shape matches it.
    if mask.shape[:2] != frame.shape[:2]:
        return overlay_frame

    # Convert any non-zero mask values into a simple boolean selection.
    selected_pixels = mask > 0
    if not np.any(selected_pixels):
        return overlay_frame

    # Paint the selected pixels with the requested overlay colour.
    coloured_layer = frame.copy()
    coloured_layer[selected_pixels] = colour_bgr

    # Blend only the selected pixels so unmasked image areas stay unchanged.
    blended_pixels = cv2.addWeighted(
        frame[selected_pixels],
        1.0 - alpha,
        coloured_layer[selected_pixels],
        alpha,
        0,
    )
    if blended_pixels is None:
        return overlay_frame

    overlay_frame[selected_pixels] = blended_pixels
    return overlay_frame


def build_pitwall_overlay(
    frame: np.ndarray,
    lapbar_mask: np.ndarray,
    bottle_mask: np.ndarray,
    datum_mask: np.ndarray,
) -> np.ndarray:
    """Overlay lapbar, bottle, and datum masks on one frame for visual review."""
    # Draw the lapbar mask first in green.
    overlay_frame = overlay_mask_on_frame(frame, lapbar_mask, colour_bgr=(0, 255, 0), alpha=0.55)

    # Draw the bottle mask second in magenta so overlap remains visible.
    overlay_frame = overlay_mask_on_frame(overlay_frame, bottle_mask, colour_bgr=(255, 0, 255), alpha=0.55)

    # Draw the datum mask third in cyan because it is the new reference point.
    overlay_frame = overlay_mask_on_frame(overlay_frame, datum_mask, colour_bgr=(255, 255, 0), alpha=0.55)
    return overlay_frame


def write_generated_masks_for_manifest(
    manifest_df: pd.DataFrame,
    lapbar_config: dict,
    bottle_config: dict,
    datum_config: dict,
    output_dir: Path,
    phase_index: int,
) -> pd.DataFrame:
    """Generate mask images for exported Phase 2 frames and write their paths into the manifest."""
    # Work on a copy so rerunning the cell is safe.
    updated_manifest_df = manifest_df.copy()
    output_dir.mkdir(parents=True, exist_ok=True)

    # Generate masks for every exported frame in the selected phase.
    for row_index, row in updated_manifest_df.iterrows():
        if int(row["aligned_phase_index"]) != int(phase_index):
            continue

        # Load the exported frame that the later measurement step will use.
        export_path = Path(row["export_path"])
        frame = cv2.imread(str(export_path))
        if frame is None:
            raise FileNotFoundError(f"Could not read exported frame: {export_path}")

        # Build all masks from the current Pitwall JSON configs.
        lapbar_mask = build_pitwall_mask(frame, lapbar_config)
        bottle_mask = build_pitwall_mask(frame, bottle_config)
        datum_mask = build_pitwall_mask(frame, datum_config)

        # Save masks beside the other Phase 2 output artefacts.
        lapbar_mask_path = output_dir / f"{export_path.stem}__lapbar_mask.png"
        bottle_mask_path = output_dir / f"{export_path.stem}__bottle_mask.png"
        datum_mask_path = output_dir / f"{export_path.stem}__datum_mask.png"
        cv2.imwrite(str(lapbar_mask_path), lapbar_mask)
        cv2.imwrite(str(bottle_mask_path), bottle_mask)
        cv2.imwrite(str(datum_mask_path), datum_mask)

        # Store the generated paths for the measurement chapter.
        updated_manifest_df.at[row_index, "lapbar_mask_path"] = str(lapbar_mask_path)
        updated_manifest_df.at[row_index, "bottle_mask_path"] = str(bottle_mask_path)
        updated_manifest_df.at[row_index, "datum_mask_path"] = str(datum_mask_path)

    # Return the updated manifest so the caller can save and display it.
    return updated_manifest_df

def sample_items_evenly(items: list, max_items: int) -> list:
    """Return up to max_items from a list, spaced evenly across the list."""
    # Return everything when the list is already short enough.
    if len(items) <= max_items:
        return list(items)

    # Pick stable positions across the full list so the review covers the whole run.
    selected_positions = np.linspace(0, len(items) - 1, max_items).round().astype(int)

    # Return the selected items in their original order.
    return [items[index] for index in selected_positions]


def plot_phase_cycle_mask_overlays(
    aligned_phase_grid: list[list[np.ndarray]],
    lapbar_config: dict,
    bottle_config: dict,
    datum_config: dict,
    phase_index: int,
    max_cycles_to_show: int = 12,
) -> None:
    """Show mask overlays on the selected phase across reference clip cycles."""
    # Keep only cycles that contain the requested phase.
    cycles_with_phase = [cycle_frames for cycle_frames in aligned_phase_grid if len(cycle_frames) > phase_index]
    if not cycles_with_phase:
        raise ValueError("No aligned reference cycles were available to plot.")

    # Choose a readable grid size for visual review.
    cycles_to_show = sample_items_evenly(cycles_with_phase, max_cycles_to_show)
    column_count = min(4, len(cycles_to_show))
    row_count = int(np.ceil(len(cycles_to_show) / column_count))
    figure, axes = plt.subplots(row_count, column_count, figsize=(4 * column_count, 3.4 * row_count))
    axes = np.atleast_1d(axes).ravel()

    # Render one overlay per selected cycle.
    for plot_index, cycle_frames in enumerate(cycles_to_show):
        frame = cycle_frames[phase_index]
        lapbar_mask = build_pitwall_mask(frame, lapbar_config)
        bottle_mask = build_pitwall_mask(frame, bottle_config)
        datum_mask = build_pitwall_mask(frame, datum_config)
        overlay_frame = build_pitwall_overlay(frame, lapbar_mask, bottle_mask, datum_mask)

        axes[plot_index].imshow(cv2.cvtColor(overlay_frame, cv2.COLOR_BGR2RGB))
        axes[plot_index].set_title(f"Cycle {plot_index:02d}, phase {phase_index:02d}")
        axes[plot_index].axis("off")

    # Hide unused axes when the number of cycles is not a square grid.
    for axis in axes[len(cycles_to_show):]:
        axis.axis("off")

    # Add a short legend in the title rather than drawing extra chart elements.
    figure.suptitle("Pitwall config masks over Phase 2 reference cycles: green=lapbar, magenta=bottle, cyan=datum")
    plt.tight_layout()
    plt.show()

def find_mask_roi_records(mask_image: np.ndarray) -> list[dict]:
    """Find connected white-pixel regions and describe each ROI with minAreaRect."""
    # Convert every non-zero mask pixel to a simple binary image.
    binary_mask = np.where(mask_image > 0, 255, 0).astype(np.uint8)

    # Label each connected white region as one ROI.
    roi_count_with_background, labels, stats, _centroids = cv2.connectedComponentsWithStats(
        binary_mask,
        connectivity=8,
    )

    # Build one readable record per ROI and skip the background label at index 0.
    roi_records = []
    for roi_label in range(1, roi_count_with_background):
        left = int(stats[roi_label, cv2.CC_STAT_LEFT])
        top = int(stats[roi_label, cv2.CC_STAT_TOP])
        width = int(stats[roi_label, cv2.CC_STAT_WIDTH])
        height = int(stats[roi_label, cv2.CC_STAT_HEIGHT])
        area = int(stats[roi_label, cv2.CC_STAT_AREA])

        # Ignore empty regions even though OpenCV should not normally return them.
        if area == 0:
            continue

        # Isolate this connected ROI so its contour can be measured directly.
        roi_mask = np.where(labels == roi_label, 255, 0).astype(np.uint8)
        contours, _hierarchy = cv2.findContours(roi_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        # Use the largest contour if OpenCV returns more than one for this ROI.
        largest_contour = max(contours, key=cv2.contourArea)

        # Use minAreaRect so the ROI center follows the rotated bottle mask shape.
        min_area_rect = cv2.minAreaRect(largest_contour)
        center_x, center_y = min_area_rect[0]
        rect_width, rect_height = min_area_rect[1]
        rect_angle = min_area_rect[2]
        box_points = cv2.boxPoints(min_area_rect)

        # Store both the bounding box and rotated-rectangle geometry for review and drawing.
        roi_records.append(
            {
                "left": left,
                "top": top,
                "width": width,
                "height": height,
                "area": area,
                "center_x": float(center_x),
                "center_y": float(center_y),
                "min_area_rect_width": float(rect_width),
                "min_area_rect_height": float(rect_height),
                "min_area_rect_angle": float(rect_angle),
                "min_area_rect": min_area_rect,
                "min_area_rect_box_points": box_points.astype(np.float32),
            }
        )

    return roi_records


def measure_lapbar_position_from_mask(mask_path: str) -> tuple[float, float]:
    """Measure the lap bar as the topmost, then rightmost, mask pixel."""
    mask_image = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_image is None:
        return np.nan, np.nan

    # Find every white pixel in the mask.
    ys, xs = np.where(mask_image > 0)
    if len(xs) == 0:
        return np.nan, np.nan

    # First find the largest y so the bottom edge takes precedence.
    maximum_y = np.max(ys)
    bottommost_pixel_indices = np.where(ys == maximum_y)[0]

    # Then choose the largest x among only the bottommost pixels.
    selected_pixel_index = bottommost_pixel_indices[np.argmax(xs[bottommost_pixel_indices])]

    # Return the selected pixel in image coordinates.
    return float(xs[selected_pixel_index]), float(ys[selected_pixel_index])


def get_middle_bottle_x_range_from_config(bottle_config: dict | None) -> tuple[float, float] | None:
    """Read the configured middle-bottle x-range from the bottle Pitwall config."""
    # Stop early when no config was provided by an older call site.
    if not bottle_config:
        return None

    # Read enabled rectangle ROIs from the Pitwall config.
    roi_settings = bottle_config.get("roi", {})
    active_rois = roi_settings.get("active_rois", [])
    enabled_rectangle_rois = [
        roi_record
        for roi_record in active_rois
        if roi_record.get("enabled", True) and roi_record.get("shape", "rectangle") == "rectangle"
    ]
    if not enabled_rectangle_rois:
        return None

    # Prefer an explicitly named middle ROI if bottles_v2.json contains one.
    named_middle_rois = [
        roi_record
        for roi_record in enabled_rectangle_rois
        if "middle" in str(roi_record.get("name", "")).lower()
    ]
    if named_middle_rois:
        selected_roi = named_middle_rois[0]
    else:
        # Fall back to the horizontal middle ROI when multiple bottle ROIs are configured.
        sorted_rois = sorted(enabled_rectangle_rois, key=lambda roi_record: float(roi_record.get("x", 0)))
        selected_roi = sorted_rois[len(sorted_rois) // 2]

    # Convert the selected ROI rectangle into an inclusive x-range.
    x_min = float(selected_roi.get("x", 0))
    x_max = x_min + float(selected_roi.get("width", 0))
    if x_max <= x_min:
        return None
    return x_min, x_max


def select_middle_bottle_roi_record(
    roi_records: list[dict],
    mask_shape: tuple[int, int],
    bottle_config: dict | None = None,
) -> dict | None:
    """Select the bottle ROI whose centre is inside the configured middle-bottle x-range."""
    # Stop early when the mask did not produce any bottle candidates.
    if not roi_records:
        return None

    # Use bottles_v2.json to restrict measurement to the configured middle-bottle x-range.
    middle_x_range = get_middle_bottle_x_range_from_config(bottle_config)
    if middle_x_range is not None:
        x_min, x_max = middle_x_range
        roi_records = [
            roi_record
            for roi_record in roi_records
            if x_min <= float(roi_record["center_x"]) <= x_max
        ]
        if not roi_records:
            return None

    # Sort the remaining candidates from left to right.
    sorted_roi_records = sorted(roi_records, key=lambda roi_record: float(roi_record["center_x"]))

    # For an odd number of candidates, the middle index is the horizontal median.
    if len(sorted_roi_records) % 2 == 1:
        middle_index = len(sorted_roi_records) // 2
        return sorted_roi_records[middle_index]

    # For an even number of candidates, only the two central candidates are plausible.
    left_middle_index = (len(sorted_roi_records) // 2) - 1
    right_middle_index = len(sorted_roi_records) // 2
    middle_candidates = [sorted_roi_records[left_middle_index], sorted_roi_records[right_middle_index]]

    # Choose the central candidate closest to the frame centre; if tied, prefer the lower one.
    _mask_height, mask_width = mask_shape[:2]
    mask_center_x = mask_width / 2.0
    return min(
        middle_candidates,
        key=lambda roi_record: (
            abs(float(roi_record["center_x"]) - mask_center_x),
            -float(roi_record["center_y"]),
        ),
    )


def measure_bottle_position_from_mask(mask_path: str, bottle_config: dict | None = None) -> tuple[float, float, int, bool, bool]:
    """Measure the configured middle bottle ROI from the bottle mask."""
    middle_roi_record, bottle_roi_count, bottle_measurement_valid, middle_roi_full_size = measure_bottle_roi_geometry_from_mask(mask_path, bottle_config)
    if not bottle_measurement_valid or middle_roi_record is None:
        return np.nan, np.nan, bottle_roi_count, False, middle_roi_full_size
    return (float(middle_roi_record["center_x"]), float(middle_roi_record["center_y"]), bottle_roi_count, True, middle_roi_full_size)

def is_min_area_rect_fully_visible(roi_record: dict, frame_shape: tuple[int, int], margin: float = 1.0) -> bool:
    """Return True when the minAreaRect box is fully inside the frame bounds."""
    frame_height, frame_width = frame_shape[:2]
    box_points = np.asarray(roi_record["min_area_rect_box_points"], dtype=np.float32)
    if box_points.size == 0:
        return False
    min_x = float(np.min(box_points[:, 0]))
    max_x = float(np.max(box_points[:, 0]))
    min_y = float(np.min(box_points[:, 1]))
    max_y = float(np.max(box_points[:, 1]))
    return min_x >= margin and min_y >= margin and max_x <= (frame_width - 1 - margin) and max_y <= (frame_height - 1 - margin)


def measure_bottle_roi_geometry_from_mask(mask_path: str, bottle_config: dict | None = None) -> tuple[dict | None, int, bool, bool]:
    """Return the configured middle bottle ROI when a valid candidate is present."""
    mask_image = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_image is None:
        return None, 0, False, False

    # Extract every connected bottle-mask candidate from the binary mask.
    roi_records = find_mask_roi_records(mask_image)
    bottle_roi_count = len(roi_records)

    # Select the target bottle from the configured middle-bottle x-range.
    middle_roi_record = select_middle_bottle_roi_record(roi_records, mask_image.shape, bottle_config)
    if middle_roi_record is None:
        return None, bottle_roi_count, False, False

    # The selected bottle must still be a visible full-size ROI before it can be measured.
    middle_roi_full_size = (
        middle_roi_record["min_area_rect_width"] > 1.0
        and middle_roi_record["min_area_rect_height"] > 1.0
        and is_min_area_rect_fully_visible(middle_roi_record, mask_image.shape)
    )

    return middle_roi_record, bottle_roi_count, middle_roi_full_size, middle_roi_full_size




def measure_centroid_position_from_mask(mask_path: str) -> tuple[float, float]:
    """Measure the centroid of all white pixels in a mask."""
    mask_image = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_image is None:
        return np.nan, np.nan

    # Find every white pixel in the mask.
    ys, xs = np.where(mask_image > 0)
    if len(xs) == 0:
        return np.nan, np.nan

    # Use the average white-pixel position as the mask centroid.
    return float(np.mean(xs)), float(np.mean(ys))


def compute_xy_offset(
    target_x: float,
    target_y: float,
    reference_x: float,
    reference_y: float,
) -> tuple[float, float]:
    """Return a target point offset relative to a reference point."""
    if pd.isna(target_x) or pd.isna(target_y) or pd.isna(reference_x) or pd.isna(reference_y):
        return np.nan, np.nan
    return float(target_x - reference_x), float(target_y - reference_y)


def compute_euclidean_distance(offset_x: float, offset_y: float) -> float:
    """Return the Euclidean distance for an x/y offset."""
    if pd.isna(offset_x) or pd.isna(offset_y):
        return np.nan
    return float(np.hypot(offset_x, offset_y))


def compute_lapbar_to_bottle_offset(
    lapbar_x: float,
    lapbar_y: float,
    bottle_x: float,
    bottle_y: float,
) -> tuple[float, float]:
    """Return the lap bar offset relative to the bottle."""
    return compute_xy_offset(lapbar_x, lapbar_y, bottle_x, bottle_y)


def save_position_measurements(measurement_df: pd.DataFrame, output_path: Path) -> Path:
    """Save the measured per-frame positions to CSV."""
    measurement_df.to_csv(output_path, index=False)
    return output_path


In [3]:
def populate_manifest_measurements(measurement_df: pd.DataFrame, bottle_config: dict) -> pd.DataFrame:
    """Fill one manifest DataFrame with lapbar, bottle, and datum measurements."""
    updated_df = measurement_df.copy()

    # Measure each row from the generated mask paths.
    for row_index, row in updated_df.iterrows():
        lapbar_mask_path = str(row.get("lapbar_mask_path", "")).strip()
        bottle_mask_path = str(row.get("bottle_mask_path", "")).strip()
        datum_mask_path = str(row.get("datum_mask_path", "")).strip()

        if lapbar_mask_path:
            lapbar_x, lapbar_y = measure_lapbar_position_from_mask(lapbar_mask_path)
        else:
            lapbar_x, lapbar_y = np.nan, np.nan

        if bottle_mask_path:
            bottle_x, bottle_y, bottle_roi_count, bottle_measurement_valid, bottle_middle_roi_full_size = measure_bottle_position_from_mask(bottle_mask_path, bottle_config)
        else:
            bottle_x, bottle_y = np.nan, np.nan
            bottle_roi_count = 0
            bottle_measurement_valid = False
            bottle_middle_roi_full_size = False

        if datum_mask_path:
            datum_x, datum_y = measure_centroid_position_from_mask(datum_mask_path)
        else:
            datum_x, datum_y = np.nan, np.nan

        # Convert the measured points into datum-relative offsets.
        bottle_datum_x, bottle_datum_y = compute_xy_offset(bottle_x, bottle_y, datum_x, datum_y)
        bottle_datum_euclidean = compute_euclidean_distance(bottle_datum_x, bottle_datum_y)
        lapbar_datum_x, lapbar_datum_y = compute_xy_offset(lapbar_x, lapbar_y, datum_x, datum_y)
        lapbar_datum_euclidean = compute_euclidean_distance(lapbar_datum_x, lapbar_datum_y)

        updated_df.at[row_index, "lapbar_x_from_datum"] = lapbar_x
        updated_df.at[row_index, "lapbar_y_from_datum"] = lapbar_y
        updated_df.at[row_index, "bottle_x_from_datum"] = bottle_x
        updated_df.at[row_index, "bottle_y_from_datum"] = bottle_y
        updated_df.at[row_index, "datum_x_from_frame"] = datum_x
        updated_df.at[row_index, "datum_y_from_frame"] = datum_y
        updated_df.at[row_index, "bottle_roi_count"] = bottle_roi_count
        updated_df.at[row_index, "bottle_measurement_valid"] = bottle_measurement_valid
        updated_df.at[row_index, "bottle_middle_roi_full_size"] = bottle_middle_roi_full_size
        updated_df.at[row_index, "bottle_minus_datum_x"] = bottle_datum_x
        updated_df.at[row_index, "bottle_minus_datum_y"] = bottle_datum_y
        updated_df.at[row_index, "bottle_from_datum_euclidean"] = bottle_datum_euclidean
        updated_df.at[row_index, "lapbar_minus_datum_x"] = lapbar_datum_x
        updated_df.at[row_index, "lapbar_minus_datum_y"] = lapbar_datum_y
        updated_df.at[row_index, "lapbar_from_datum_euclidean"] = lapbar_datum_euclidean

    return updated_df


def filter_valid_overlay_rows(measurement_df: pd.DataFrame, required_columns: list[str]) -> pd.DataFrame:
    """Keep only rows that have every plotted measurement and pass bottle validity checks."""
    filtered_df = measurement_df.dropna(subset=required_columns).copy()

    # Keep the same bottle validity rules used by the overlay review cells.
    if "bottle_measurement_valid" in filtered_df.columns:
        filtered_df = filtered_df[filtered_df["bottle_measurement_valid"].astype(bool)].copy()
    if "bottle_middle_roi_full_size" in filtered_df.columns:
        filtered_df = filtered_df[filtered_df["bottle_middle_roi_full_size"].astype(bool)].copy()

    return filtered_df


## 2. Load the Reference Image, Pick ROI, and Inspect the Reference Clip

**What this section does**

Load the reference image that represents the visual state you want to call aligned `phase 0`, then use the reference clip to pick a single ROI that will be reused for both alignment and measurement export.

**Existing repo functions used here**

- `read_video()`
- `select_roi_on_frames()`


In [4]:
reference_image_path = resolve_reference_image_path(reference_dir, reference_image_path)
reference_image = load_reference_image(reference_image_path)
video_paths = list_video_paths(video_dir, max_items=max_videos)
reference_video_paths = list_video_paths(reference_dir, max_items=1)

if not reference_video_paths:
    raise FileNotFoundError(
        f"No reference clip was found in: {reference_dir}. Add one video so Chapter 2 and Chapter 3 can use it."
    )

reference_video_path = reference_video_paths[0]

# Load the reference clip at native resolution so the preview matches the reference image geometry.
reference_frames, reference_fps = read_video(reference_video_path, max_frames=phase_max_frames, resize=None)

if use_measurement_roi:
    if roi_json_path.exists():
        measurement_roi = load_roi(roi_json_path)
        print(f"Loaded existing ROI from: {roi_json_path}")
    else:
        measurement_roi = select_roi_on_frames(reference_frames, start_index=0)
        save_roi(measurement_roi, roi_json_path)
        print(f"Saved new ROI to: {roi_json_path}")
else:
    measurement_roi = None
    print("ROI cropping is OFF. The full frame will be used.")

# Use the measurement ROI as the phase-detection ROI as well.
# Roi.frame_width/frame_height in measurement_roi.json lets VideoModule scale it to any decoded frame size.
# Keep phase detection separate from measurement.
# measurement_roi is too tight for cycle detection on this clip, so phase_roi stays full-frame unless set above.
cfg = cfg.replace(phase_roi=phase_detection_roi)
print(f"Measurement ROI: {measurement_roi}")
print(f"Phase ROI: {cfg.phase_roi}")

# Show the reference image and the reference clip side by side.
figure, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(reference_image, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Reference image: {reference_image_path.name}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(reference_frames[0], cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Reference clip: {reference_video_path.name}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"Stop clips found: {len(video_paths)}")
print(f"Reference clip FPS: {reference_fps:.2f}")
print(f"Use ROI: {use_measurement_roi}")
print(f"ROI: {measurement_roi}")


Loaded existing ROI from: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\config\measurement_roi.json
Measurement ROI: Roi(x=628, y=9, w=929, h=651, frame_width=1920, frame_height=1000)
Phase ROI: None


C:\Users\TimKitchen\AppData\Local\Temp\ipykernel_2844\477501852.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Stop clips found: 1
Reference clip FPS: 60.00
Use ROI: True
ROI: Roi(x=628, y=9, w=929, h=651, frame_width=1920, frame_height=1000)


## 3. Do Phase Awareness and Align Phase 0 to the Reference Image

**What this section does**

Detect the repeating phase cycle on the reference clip, then score each original phase column against the reference image. The phase column with the strongest NCC match becomes aligned `phase 0`.

**Existing repo functions used here**

- `run_phase_awareness()`
- `detect_dynamic_phases()` through `run_phase_awareness()`
- `resample_segments_to_phases()`
- `build_roi_phase_grid()`
- `compute_ncc_opencv()`
- `plot_dynamic_phase_overview()`
- `plot_phase_grid()`


In [5]:
reference_phase_data = detect_video_phases(reference_video_path, cfg)
reference_alignment = align_video_phases_to_reference_image(
    reference_image=reference_image,
    video_path=reference_video_path,
    cycles=reference_phase_data["phase_result"].cycles,
    cfg=cfg,
    roi=measurement_roi,
)

print(f"Reference clip: {reference_video_path.name}")
print(f"Detected cycles: {len(reference_phase_data['phase_result'].cycles)}")
print(f"Aligned phase 0 comes from original phase: {reference_alignment['best_original_phase_index']}")

display(reference_alignment["phase_score_df"])

# Show the standard phase-awareness overview first.
plot_dynamic_phase_overview(reference_phase_data["phase_result"], show=True)
plt.show()

# Show the aligned ROI phase grid so phase 0 / 1 / 2 can be checked visually.
plot_phase_grid(
    reference_alignment["aligned_phase_grid"],
    num_cycles_to_show=5,
    title="Aligned full-frame phase grid for the reference clip",
    phase_labels=list(range(cfg.num_phases)),
    highlight_phases=phases_of_interest,
)
plt.show()


c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\VideoModule\phase_detection\dynamic_phase_detection.py:1435: UserWarning: fCWT not available, using PyWavelets CWT (slower). For better performance, install fCWT: pip install fcwt
  span_regions, span_time, method = detect_frequency_regions_fcwt(


Reference clip: cortexvpu-01a-005-41884872_2026-06-19_17-35-41_933333_clip.ts
Detected cycles: 46
Aligned phase 0 comes from original phase: 8


,original_phase_index,median_ncc_to_reference_image
0,0,0.826374
1,1,0.630950
2,2,0.486366
3,3,0.362307
4,4,0.797556
5,5,0.837343
6,6,0.827215
7,7,0.857091
8,8,0.946389
9,9,0.826374


c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\VideoModule\plotting\dynamic_phase_plots.py:129: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\TimKitchen\AppData\Local\Temp\ipykernel_2844\3416955329.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\TimKitchen\AppData\Local\Temp\ipykernel_2844\3416955329.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Export Frames for Phases of Interest

**What this section does**

Choose whether to export from the reference clip or from the stop clips, align each selected clip to the reference image, and export the ROI frames for the phases of interest.

**Existing repo functions used here**

- `read_video()`
- `run_phase_awareness()`
- `detect_dynamic_phases()` through `run_phase_awareness()`
- `build_roi_phase_grid()`
- `compute_ncc_opencv()`


In [6]:
if export_clip_source == "reference_clip":
    export_video_paths = [reference_video_path]
elif export_clip_source == "stop_clips":
    export_video_paths = video_paths
else:
    raise ValueError(
        f"Unsupported export_clip_source: {export_clip_source}. Use 'reference_clip' or 'stop_clips'."
    )

all_alignment_rows = []
all_frame_records = []

for video_path in export_video_paths:
    video_phase_data = detect_video_phases(video_path, cfg)
    video_alignment = align_video_phases_to_reference_image(
        reference_image=reference_image,
        video_path=video_path,
        cycles=video_phase_data["phase_result"].cycles,
        cfg=cfg,
        roi=measurement_roi,
    )

    all_alignment_rows.append(
        {
            "video_name": video_path.name,
            "detected_cycles": len(video_phase_data["phase_result"].cycles),
            "aligned_phase_zero_source": video_alignment["best_original_phase_index"],
            "phase_zero_match_score": video_alignment["phase_score_df"]["median_ncc_to_reference_image"].max(),
            "export_clip_source": export_clip_source,
        }
    )
    all_frame_records.extend(video_alignment["frame_records"])

alignment_summary_df = pd.DataFrame(all_alignment_rows)
display(alignment_summary_df)

exported_phase_frames_df = export_phase_frames(
    frame_records=all_frame_records,
    export_dir=exported_frames_dir,
    phases_of_interest=phases_of_interest,
    samples_per_phase=export_samples_per_phase,
)
display(exported_phase_frames_df.head(20))

phase_frame_manifest_df = build_phase_frame_manifest(exported_phase_frames_df)
phase_frame_manifest_df.to_csv(measurement_template_path, index=False)

print(f"Export source: {export_clip_source}")
print(f"Clips processed for export: {len(export_video_paths)}")
print(f"Exported frames folder: {exported_frames_dir}")
print(f"Pitwall measurement template: {measurement_template_path}")


c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\VideoModule\phase_detection\dynamic_phase_detection.py:1435: UserWarning: fCWT not available, using PyWavelets CWT (slower). For better performance, install fCWT: pip install fcwt
  span_regions, span_time, method = detect_frequency_regions_fcwt(


,video_name,detected_cycles,aligned_phase_zero_source,phase_zero_match_score,export_clip_source
0,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,46,8,0.946389,reference_clip


,video_name,cycle_index,aligned_phase_index,original_phase_index,phase_zero_source_phase_index,phase_zero_match_score,keyframe_ncc_score,keyframe_ncc_margin,keyframe_frame_index,keyframe_cycle_start,keyframe_cycle_end,keyframe_candidate_count,phase_selection_method,export_path
0,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,0,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
1,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,1,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
2,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,2,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
3,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,3,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
4,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,4,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
5,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,5,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
6,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,6,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
7,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,7,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
8,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,8,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...
9,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,9,3,1,8,0.946389,0.946389,NaN,NaN,NaN,NaN,NaN,phase_zero_alignment,c:\Users\TimKitchen\OneDrive\OneDrive - Purple...


Export source: reference_clip
Clips processed for export: 1
Exported frames folder: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\frames\phase_exports
Pitwall measurement template: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\manifests\pitwall_measurement_template.csv


## 5. Review Pitwall Config Masks on Phase 2 Reference Cycles

**What this section does**

Load the Pitwall JSON configs from `pitwall_config`, apply them to the aligned Phase 2 reference cycles, and show the generated lapbar, bottle, and datum masks overlaid on the reference clip frames.

This section also writes generated mask images for the exported Phase 2 frames and saves their paths into the measurement template used by the next chapter.

**Local functions used here**

- `load_pitwall_config()`
- `build_pitwall_mask()`
- `plot_phase_cycle_mask_overlays()`
- `write_generated_masks_for_manifest()`


In [7]:
# Load the saved Pitwall JSON configs for the measured objects and datum.
lapbar_config = load_pitwall_config(lapbar_config_path)
bottle_config = load_pitwall_config(bottle_config_path)
datum_config = load_pitwall_config(datum_config_path)

# Show the masks overlaid on aligned Phase 2 cycles from the reference clip.
plot_phase_cycle_mask_overlays(
    aligned_phase_grid=reference_alignment["aligned_phase_grid"],
    lapbar_config=lapbar_config,
    bottle_config=bottle_config,
    datum_config=datum_config,
    phase_index=phase_to_review,
    max_cycles_to_show=12,
)

# Generate binary mask images for the exported Phase 2 frames.
phase_frame_manifest_df = write_generated_masks_for_manifest(
    manifest_df=phase_frame_manifest_df,
    lapbar_config=lapbar_config,
    bottle_config=bottle_config,
    datum_config=datum_config,
    output_dir=pitwall_mask_output_dir,
    phase_index=phase_to_review,
)

# Save the updated template so the measurement chapter can read the generated mask paths.
phase_frame_manifest_df.to_csv(measurement_template_path, index=False)

# Show the rows that now have generated mask paths.
display(phase_frame_manifest_df.head(20))

print(f"Pitwall configs loaded from: {pitwall_config_dir}")
print(f"Generated masks folder: {pitwall_mask_output_dir}")
print(f"Updated measurement template: {measurement_template_path}")


C:\Users\TimKitchen\AppData\Local\Temp\ipykernel_2844\3058434210.py:822: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,video_name,cycle_index,aligned_phase_index,original_phase_index,phase_zero_source_phase_index,phase_zero_match_score,keyframe_ncc_score,keyframe_ncc_margin,keyframe_frame_index,keyframe_cycle_start,...,datum_y_from_frame,bottle_roi_count,bottle_measurement_valid,bottle_middle_roi_full_size,bottle_minus_datum_x,bottle_minus_datum_y,bottle_from_datum_euclidean,lapbar_minus_datum_x,lapbar_minus_datum_y,lapbar_from_datum_euclidean
0,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,0,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
1,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,1,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
2,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,2,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
3,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,3,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
4,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,4,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
5,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,5,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
6,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,6,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
7,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,7,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
8,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,8,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN
9,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,9,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN


Pitwall configs loaded from: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\pitwall_config
Generated masks folder: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\masks\pitwall
Updated measurement template: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\manifests\pitwall_measurement_template.csv


## 6. Export Per-Frame Measurements to CSV

**What this section does**

Fill in the Pitwall mask paths for each exported frame, then calculate per-frame lap bar, bottle, and datum positions. The bottle position uses the minAreaRect centroid of the horizontally middle detected bottle ROI whenever that ROI is visible and full sized. The final measurements are datum-to-lapbar and datum-to-bottle x/y offsets plus Euclidean distances.

**Local functions used here**

- `build_phase_frame_manifest()`
- `measure_lapbar_position_from_mask()`
- `measure_bottle_position_from_mask()`
- `measure_centroid_position_from_mask()`
- `compute_xy_offset()`
- `compute_euclidean_distance()`
- `save_position_measurements()`


In [8]:
# Update the template CSV after Pitwall has written lapbar_mask_path, bottle_mask_path, and datum_mask_path.
pitwall_measurement_df = pd.read_csv(measurement_template_path)

def ensure_measurement_column(column_name: str, default_value) -> None:
    """Add a measurement column when the template was created by an older notebook version."""
    if column_name not in pitwall_measurement_df.columns:
        pitwall_measurement_df[column_name] = default_value


ensure_measurement_column("datum_mask_path", "")
ensure_measurement_column("datum_x_from_frame", np.nan)
ensure_measurement_column("datum_y_from_frame", np.nan)
ensure_measurement_column("bottle_roi_count", np.nan)
ensure_measurement_column("bottle_measurement_valid", False)
ensure_measurement_column("bottle_middle_roi_full_size", False)
ensure_measurement_column("bottle_minus_datum_x", np.nan)
ensure_measurement_column("bottle_minus_datum_y", np.nan)
ensure_measurement_column("bottle_from_datum_euclidean", np.nan)
ensure_measurement_column("lapbar_minus_datum_x", np.nan)
ensure_measurement_column("lapbar_minus_datum_y", np.nan)
ensure_measurement_column("lapbar_from_datum_euclidean", np.nan)

# Reuse the shared manifest measurement helper so Chapter 8 can run the same logic.
pitwall_measurement_df = populate_manifest_measurements(pitwall_measurement_df, bottle_config)

save_position_measurements(pitwall_measurement_df, measurements_output_path)
display(pitwall_measurement_df.head(20))
print(f"Saved per-frame measurements to: {measurements_output_path}")


,video_name,cycle_index,aligned_phase_index,original_phase_index,phase_zero_source_phase_index,phase_zero_match_score,keyframe_ncc_score,keyframe_ncc_margin,keyframe_frame_index,keyframe_cycle_start,...,datum_y_from_frame,bottle_roi_count,bottle_measurement_valid,bottle_middle_roi_full_size,bottle_minus_datum_x,bottle_minus_datum_y,bottle_from_datum_euclidean,lapbar_minus_datum_x,lapbar_minus_datum_y,lapbar_from_datum_euclidean
0,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,0,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,3.0,True,True,-117.420776,195.774811,228.288009,371.5,210.0,426.746119
1,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,1,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,6.0,True,True,-101.839966,207.380005,231.036458,265.5,191.0,327.064596
2,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,2,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,3.0,False,False,NaN,NaN,NaN,403.5,191.0,446.422726
3,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,3,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,3.0,True,True,-85.551086,192.609497,210.754375,419.5,203.0,466.035675
4,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,4,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,5.0,True,True,-110.545532,197.739990,226.542310,387.5,207.0,439.323628
5,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,5,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,6.0,False,False,NaN,NaN,NaN,299.5,210.0,365.787165
6,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,6,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,3.0,True,True,-111.465637,198.016846,227.233931,387.5,205.0,438.384820
7,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,7,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,4.0,True,True,-93.345520,196.155457,217.233398,411.5,210.0,461.987283
8,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,8,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,4.0,True,True,-86.228333,201.188660,218.888561,403.5,193.0,447.282070
9,cortexvpu-01a-005-41884872_2026-06-19_17-35-41...,9,3,1,8,0.946389,0.946389,NaN,NaN,NaN,...,68.0,3.0,True,True,-91.991943,194.940125,215.555491,395.5,199.0,442.742871


Saved per-frame measurements to: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\measurements\pitwall_position_measurements.csv


## 7. Overlay Datum-Based Distances on Frames

**What this section does**

Start with the raw frame-level measurements and draw both datum-based relationships directly on each exported frame: datum to lap bar, and datum to bottle. The text is kept in one stacked panel so both measurements are visible without overlapping labels, and the selected horizontally middle bottle minAreaRect is drawn so the plotted geometry matches the bottle calculation.

**Local functions used here**

- `save_position_measurements()`


In [9]:
measured_frames_df = pd.read_csv(measurements_output_path)

# Chapter 7 is a review plot, so keep every exported cycle/frame.
# Bottle measurements may be invalid on some cycles; those frames still need to be shown.
review_required_columns = [
    "export_path",
    "datum_mask_path",
    "bottle_mask_path",
    "lapbar_mask_path",
    "lapbar_x_from_datum",
    "lapbar_y_from_datum",
    "datum_x_from_frame",
    "datum_y_from_frame",
    "lapbar_minus_datum_x",
    "lapbar_minus_datum_y",
    "lapbar_from_datum_euclidean",
]
overlay_df = measured_frames_df.dropna(subset=review_required_columns).copy()

chapter_7_overlay_dir = output_overlays_dir / "chapter_7_reference"
chapter_7_overlay_dir.mkdir(parents=True, exist_ok=True)


def value_is_finite(value) -> bool:
    """Return True when a numeric measurement can be safely plotted."""
    return pd.notna(value) and np.isfinite(float(value))


def draw_text_panel(frame: np.ndarray, panel_lines: list[tuple[str, tuple[int, int, int]]], top_left: tuple[int, int]) -> None:
    """Draw a stacked text panel that stays readable on larger frames."""
    frame_height, frame_width = frame.shape[:2]

    # Scale the annotation size from the frame dimensions.
    scale_reference = max(frame_width / 1920.0, frame_height / 1080.0, 1.0)
    font_face = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.9 * scale_reference
    text_thickness = max(2, int(round(2.2 * scale_reference)))
    padding = max(16, int(round(18 * scale_reference)))
    line_gap = max(10, int(round(12 * scale_reference)))
    border_thickness = max(1, int(round(1.5 * scale_reference)))

    measured_lines = []
    max_text_width = 0
    max_text_height = 0

    # Measure each line first so the panel background can fit the content.
    for line_text, line_colour in panel_lines:
        text_size, baseline = cv2.getTextSize(line_text, font_face, font_scale, text_thickness)
        text_width, text_height = text_size
        measured_lines.append(
            {
                "text": line_text,
                "colour": line_colour,
                "width": text_width,
                "height": text_height,
                "baseline": baseline,
            }
        )
        max_text_width = max(max_text_width, text_width)
        max_text_height = max(max_text_height, text_height + baseline)

    panel_width = max_text_width + (padding * 2)
    panel_height = (len(measured_lines) * max_text_height) + ((len(measured_lines) - 1) * line_gap) + (padding * 2)
    panel_x = min(max(top_left[0], 0), max(frame_width - panel_width - 1, 0))
    panel_y = min(max(top_left[1], 0), max(frame_height - panel_height - 1, 0))
    panel_bottom_right = (int(panel_x + panel_width), int(panel_y + panel_height))

    # Draw an opaque panel so the text does not compete with the frame content.
    cv2.rectangle(frame, (int(panel_x), int(panel_y)), panel_bottom_right, (0, 0, 0), thickness=-1)
    cv2.rectangle(frame, (int(panel_x), int(panel_y)), panel_bottom_right, (255, 255, 255), thickness=border_thickness)

    text_x = int(panel_x + padding)
    first_baseline_y = int(panel_y + padding + max_text_height)
    line_step = max_text_height + line_gap

    # Render each line into the panel.
    for line_index, measured_line in enumerate(measured_lines):
        text_y = first_baseline_y + (line_index * line_step)
        cv2.putText(
            frame,
            measured_line["text"],
            (text_x, text_y),
            font_face,
            font_scale,
            measured_line["colour"],
            text_thickness,
            cv2.LINE_AA,
        )


def draw_distance_overlay(frame: np.ndarray, row: pd.Series) -> tuple[np.ndarray, float, float | None]:
    """Draw datum-to-lapbar always, and datum-to-bottle only when bottle measurement is valid."""
    overlay_frame = frame.copy()
    frame_height, frame_width = overlay_frame.shape[:2]

    # Scale markers and lines from the frame dimensions.
    scale_reference = max(frame_width / 1920.0, frame_height / 1080.0, 1.0)
    point_radius = max(8, int(round(10 * scale_reference)))
    line_thickness = max(3, int(round(3.5 * scale_reference)))
    box_thickness = max(2, int(round(3 * scale_reference)))

    datum_point = (int(round(row["datum_x_from_frame"])), int(round(row["datum_y_from_frame"])))
    lapbar_point = (int(round(row["lapbar_x_from_datum"])), int(round(row["lapbar_y_from_datum"])))
    bottle_roi_record, bottle_roi_count, bottle_measurement_valid, bottle_middle_roi_full_size = measure_bottle_roi_geometry_from_mask(str(row["bottle_mask_path"]), bottle_config)

    has_bottle_measurement = all(
        value_is_finite(row.get(column_name, np.nan))
        for column_name in [
            "bottle_x_from_datum",
            "bottle_y_from_datum",
            "bottle_minus_datum_x",
            "bottle_minus_datum_y",
            "bottle_from_datum_euclidean",
        ]
    )

    datum_to_lapbar_x = float(row["lapbar_minus_datum_x"])
    datum_to_lapbar_y = float(row["lapbar_minus_datum_y"])
    datum_to_lapbar_distance = float(row["lapbar_from_datum_euclidean"])

    # Draw the datum/lapbar reference points and measurement line for every review frame.
    cv2.circle(overlay_frame, datum_point, radius=point_radius, color=(255, 255, 0), thickness=-1)
    cv2.circle(overlay_frame, lapbar_point, radius=point_radius, color=(0, 255, 0), thickness=-1)
    cv2.line(overlay_frame, datum_point, lapbar_point, color=(0, 255, 255), thickness=line_thickness)

    panel_lines = [
        ("datum to lapbar", (0, 255, 255)),
        (f"euclidean: {datum_to_lapbar_distance:.1f} px", (255, 255, 255)),
        (f"x: {datum_to_lapbar_x:.1f} px, y: {datum_to_lapbar_y:.1f} px", (255, 255, 255)),
        (f"lapbar: ({row['lapbar_x_from_datum']:.1f}, {row['lapbar_y_from_datum']:.1f})", (255, 255, 255)),
        ("", (255, 255, 255)),
        ("datum to bottle", (0, 165, 255)),
    ]

    datum_to_bottle_distance = None
    if has_bottle_measurement:
        bottle_point = (int(round(row["bottle_x_from_datum"])), int(round(row["bottle_y_from_datum"])))
        datum_to_bottle_x = float(row["bottle_minus_datum_x"])
        datum_to_bottle_y = float(row["bottle_minus_datum_y"])
        datum_to_bottle_distance = float(row["bottle_from_datum_euclidean"])

        cv2.circle(overlay_frame, bottle_point, radius=point_radius, color=(255, 0, 255), thickness=-1)
        cv2.line(overlay_frame, datum_point, bottle_point, color=(0, 165, 255), thickness=line_thickness)
        if bottle_measurement_valid and bottle_middle_roi_full_size and bottle_roi_record is not None:
            bottle_box_points = np.round(bottle_roi_record["min_area_rect_box_points"]).astype(np.int32)
            cv2.polylines(overlay_frame, [bottle_box_points], isClosed=True, color=(255, 0, 255), thickness=box_thickness)

        panel_lines.extend(
            [
                (f"euclidean: {datum_to_bottle_distance:.1f} px", (255, 255, 255)),
                (f"x: {datum_to_bottle_x:.1f} px, y: {datum_to_bottle_y:.1f} px", (255, 255, 255)),
                (f"bottle: ({row['bottle_x_from_datum']:.1f}, {row['bottle_y_from_datum']:.1f})", (255, 255, 255)),
            ]
        )
    else:
        panel_lines.append(("not measured on this cycle", (80, 180, 255)))

    panel_lines.extend(
        [
            (f"datum: ({row['datum_x_from_frame']:.1f}, {row['datum_y_from_frame']:.1f})", (255, 255, 255)),
            (f"bottle rois: {bottle_roi_count}", (255, 255, 255)),
            (f"middle roi full size: {bool(bottle_middle_roi_full_size)}", (255, 255, 255)),
        ]
    )
    draw_text_panel(overlay_frame, panel_lines, top_left=(24, 24))
    return overlay_frame, datum_to_lapbar_distance, datum_to_bottle_distance


if overlay_df.empty:
    print("No review rows were found yet. Run Chapters 5 and 6 after generating the Pitwall masks, then rerun this cell.")
else:
    overlay_records = []
    for _row_index, row in overlay_df.iterrows():
        frame_path = Path(row["export_path"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            print(f"Skipped unreadable frame: {frame_path}")
            continue
        overlay_frame, datum_to_lapbar_distance, datum_to_bottle_distance = draw_distance_overlay(frame, row)
        overlay_path = chapter_7_overlay_dir / f"{frame_path.stem}__chapter_7_overlay.png"
        cv2.imwrite(str(overlay_path), overlay_frame)
        overlay_records.append(
            {
                "row": row,
                "overlay_frame": overlay_frame,
                "overlay_path": overlay_path,
                "datum_to_lapbar_distance": datum_to_lapbar_distance,
                "datum_to_bottle_distance": datum_to_bottle_distance,
            }
        )

    if not overlay_records:
        print("No readable frames were available for overlay review.")
    else:
        # Keep the grid shallow so each frame gets enough screen area.
        column_count = min(2, len(overlay_records))
        row_count = int(np.ceil(len(overlay_records) / column_count))
        figure, axes = plt.subplots(
            row_count,
            column_count,
            figsize=(8.5 * column_count, 6.6 * row_count),
            dpi=180,
        )
        axes = np.atleast_1d(axes).ravel()

        for plot_index, overlay_record in enumerate(overlay_records):
            row = overlay_record["row"]
            overlay_frame = overlay_record["overlay_frame"]
            datum_to_lapbar_distance = overlay_record["datum_to_lapbar_distance"]
            datum_to_bottle_distance = overlay_record["datum_to_bottle_distance"]
            bottle_title = "not measured" if datum_to_bottle_distance is None else f"{datum_to_bottle_distance:.1f}px"
            axes[plot_index].imshow(cv2.cvtColor(overlay_frame, cv2.COLOR_BGR2RGB))
            axes[plot_index].set_title(
                f"cycle {int(row['cycle_index'])}, phase {int(row['aligned_phase_index'])}, "
                f"lapbar {datum_to_lapbar_distance:.1f}px, bottle {bottle_title}",
                fontsize=13,
            )
            axes[plot_index].axis("off")

        for axis in axes[len(overlay_records):]:
            axis.axis("off")

        plt.tight_layout(pad=1.2)
        plt.show()
        print(f"Saved Chapter 7 reference overlays to: {chapter_7_overlay_dir}")


C:\Users\TimKitchen\AppData\Local\Temp\ipykernel_2844\3144205918.py:224: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved Chapter 7 reference overlays to: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\overlays\chapter_7_reference


## 7A. Overlay Datum-Based Distances on Rotated Frames with Masks

**What this section does**

Recreate the Chapter 7 review plot, but rotate the frame, masks, and plotted measurement geometry by 180 degrees before display. The same metrics and stacked text panel are kept so the rotated view can be compared directly with Chapter 7.

**Local functions used here**

- `save_position_measurements()`


In [10]:
chapter_7a_frames_df = pd.read_csv(measurements_output_path)

# Show every valid cycle/frame in the rotated Chapter 7A review plot.
chapter_7a_required_columns = [
    "export_path",
    "datum_mask_path",
    "bottle_mask_path",
    "lapbar_mask_path",
    "lapbar_x_from_datum",
    "lapbar_y_from_datum",
    "datum_x_from_frame",
    "datum_y_from_frame",
    "lapbar_minus_datum_x",
    "lapbar_minus_datum_y",
    "lapbar_from_datum_euclidean",
]
chapter_7a_overlay_df = chapter_7a_frames_df.dropna(subset=chapter_7a_required_columns).copy()

chapter_7a_overlay_dir = output_overlays_dir / "chapter_7a_reference"
chapter_7a_overlay_dir.mkdir(parents=True, exist_ok=True)

def rotate_point_180(point_xy: tuple[int, int], frame_width: int, frame_height: int) -> tuple[int, int]:
    """Rotate one pixel coordinate by 180 degrees inside the frame bounds."""
    point_x, point_y = point_xy
    rotated_x = frame_width - 1 - point_x
    rotated_y = frame_height - 1 - point_y
    return int(rotated_x), int(rotated_y)


def rotate_box_points_180(box_points: np.ndarray, frame_width: int, frame_height: int) -> np.ndarray:
    """Rotate a set of contour points by 180 degrees."""
    rotated_points = box_points.astype(np.float32).copy()
    rotated_points[:, 0] = frame_width - 1 - rotated_points[:, 0]
    rotated_points[:, 1] = frame_height - 1 - rotated_points[:, 1]
    return np.round(rotated_points).astype(np.int32)


def load_mask_image(mask_path_value: str) -> np.ndarray | None:
    """Load one mask image as grayscale when the path exists."""
    mask_path_text = str(mask_path_value).strip()
    if not mask_path_text:
        return None

    mask_image = cv2.imread(mask_path_text, cv2.IMREAD_GRAYSCALE)
    if mask_image is None:
        return None

    return mask_image


def add_coloured_mask_overlay(frame: np.ndarray, mask_image: np.ndarray | None, mask_colour: tuple[int, int, int], alpha: float = 0.30) -> None:
    """Blend one binary mask into the frame using a fixed colour."""
    if mask_image is None:
        return

    mask_pixels = mask_image > 0
    if not np.any(mask_pixels):
        return

    colour_layer = np.zeros_like(frame, dtype=np.uint8)
    colour_layer[mask_pixels] = mask_colour
    blended_frame = cv2.addWeighted(frame, 1.0, colour_layer, alpha, 0.0)
    frame[mask_pixels] = blended_frame[mask_pixels]


def draw_rotated_distance_overlay(frame: np.ndarray, row: pd.Series) -> tuple[np.ndarray, float, float | None]:
    """Draw the Chapter 7 measurements on a 180-degree rotated frame with mask overlays."""
    rotated_frame = cv2.rotate(frame.copy(), cv2.ROTATE_180)
    frame_height, frame_width = rotated_frame.shape[:2]

    # Scale markers and lines from the rotated frame dimensions.
    scale_reference = max(frame_width / 1920.0, frame_height / 1080.0, 1.0)
    point_radius = max(8, int(round(10 * scale_reference)))
    line_thickness = max(3, int(round(3.5 * scale_reference)))
    box_thickness = max(2, int(round(3 * scale_reference)))

    # Load and rotate the masks before drawing them on the frame.
    datum_mask_image = load_mask_image(row["datum_mask_path"])
    bottle_mask_image = load_mask_image(row["bottle_mask_path"])
    lapbar_mask_image = load_mask_image(row["lapbar_mask_path"])

    if datum_mask_image is not None:
        datum_mask_image = cv2.rotate(datum_mask_image, cv2.ROTATE_180)
    if bottle_mask_image is not None:
        bottle_mask_image = cv2.rotate(bottle_mask_image, cv2.ROTATE_180)
    if lapbar_mask_image is not None:
        lapbar_mask_image = cv2.rotate(lapbar_mask_image, cv2.ROTATE_180)

    add_coloured_mask_overlay(rotated_frame, datum_mask_image, (255, 255, 0))
    add_coloured_mask_overlay(rotated_frame, bottle_mask_image, (255, 0, 255))
    add_coloured_mask_overlay(rotated_frame, lapbar_mask_image, (0, 255, 0))

    # Rotate the measured datum/lapbar points so the geometry still matches the displayed frame.
    original_datum_point = (int(round(row["datum_x_from_frame"])), int(round(row["datum_y_from_frame"])))
    original_lapbar_point = (int(round(row["lapbar_x_from_datum"])), int(round(row["lapbar_y_from_datum"])))
    datum_point = rotate_point_180(original_datum_point, frame_width, frame_height)
    lapbar_point = rotate_point_180(original_lapbar_point, frame_width, frame_height)

    bottle_roi_record, bottle_roi_count, bottle_measurement_valid, bottle_middle_roi_full_size = measure_bottle_roi_geometry_from_mask(str(row["bottle_mask_path"]), bottle_config)
    has_bottle_measurement = all(
        value_is_finite(row.get(column_name, np.nan))
        for column_name in [
            "bottle_x_from_datum",
            "bottle_y_from_datum",
            "bottle_minus_datum_x",
            "bottle_minus_datum_y",
            "bottle_from_datum_euclidean",
        ]
    )

    datum_to_lapbar_x = float(row["lapbar_minus_datum_x"])
    datum_to_lapbar_y = float(row["lapbar_minus_datum_y"])
    datum_to_lapbar_distance = float(row["lapbar_from_datum_euclidean"])

    # Draw the rotated datum/lapbar reference points and measurement line for every frame.
    cv2.circle(rotated_frame, datum_point, radius=point_radius, color=(255, 255, 0), thickness=-1)
    cv2.circle(rotated_frame, lapbar_point, radius=point_radius, color=(0, 255, 0), thickness=-1)
    cv2.line(rotated_frame, datum_point, lapbar_point, color=(0, 255, 255), thickness=line_thickness)

    panel_lines = [
        ("datum to lapbar", (0, 255, 255)),
        (f"euclidean: {datum_to_lapbar_distance:.1f} px", (255, 255, 255)),
        (f"x: {datum_to_lapbar_x:.1f} px, y: {datum_to_lapbar_y:.1f} px", (255, 255, 255)),
        (f"lapbar: ({row['lapbar_x_from_datum']:.1f}, {row['lapbar_y_from_datum']:.1f})", (255, 255, 255)),
        ("", (255, 255, 255)),
        ("datum to bottle", (0, 165, 255)),
    ]

    datum_to_bottle_distance = None
    if has_bottle_measurement:
        original_bottle_point = (int(round(row["bottle_x_from_datum"])), int(round(row["bottle_y_from_datum"])))
        bottle_point = rotate_point_180(original_bottle_point, frame_width, frame_height)
        datum_to_bottle_x = float(row["bottle_minus_datum_x"])
        datum_to_bottle_y = float(row["bottle_minus_datum_y"])
        datum_to_bottle_distance = float(row["bottle_from_datum_euclidean"])

        cv2.circle(rotated_frame, bottle_point, radius=point_radius, color=(255, 0, 255), thickness=-1)
        cv2.line(rotated_frame, datum_point, bottle_point, color=(0, 165, 255), thickness=line_thickness)
        if bottle_measurement_valid and bottle_middle_roi_full_size and bottle_roi_record is not None:
            bottle_box_points = np.round(bottle_roi_record["min_area_rect_box_points"]).astype(np.int32)
            bottle_box_points = rotate_box_points_180(bottle_box_points, frame_width, frame_height)
            cv2.polylines(rotated_frame, [bottle_box_points], isClosed=True, color=(255, 0, 255), thickness=box_thickness)

        panel_lines.extend(
            [
                (f"euclidean: {datum_to_bottle_distance:.1f} px", (255, 255, 255)),
                (f"x: {datum_to_bottle_x:.1f} px, y: {datum_to_bottle_y:.1f} px", (255, 255, 255)),
                (f"bottle: ({row['bottle_x_from_datum']:.1f}, {row['bottle_y_from_datum']:.1f})", (255, 255, 255)),
            ]
        )
    else:
        panel_lines.append(("not measured on this cycle", (80, 180, 255)))

    panel_lines.extend(
        [
            (f"datum: ({row['datum_x_from_frame']:.1f}, {row['datum_y_from_frame']:.1f})", (255, 255, 255)),
            (f"bottle rois: {bottle_roi_count}", (255, 255, 255)),
            (f"middle roi full size: {bool(bottle_middle_roi_full_size)}", (255, 255, 255)),
        ]
    )
    draw_text_panel(rotated_frame, panel_lines, top_left=(24, 24))
    return rotated_frame, datum_to_lapbar_distance, datum_to_bottle_distance

if chapter_7a_overlay_df.empty:
    print("No valid rows were found yet. Run Chapters 5 and 6 after generating the Pitwall masks, then rerun this cell.")
else:
    chapter_7a_overlay_records = []

    # Build one rotated overlay per selected frame.
    for _row_index, row in chapter_7a_overlay_df.iterrows():
        frame_path = Path(row["export_path"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            print(f"Skipped unreadable frame: {frame_path}")
            continue

        rotated_overlay_frame, datum_to_lapbar_distance, datum_to_bottle_distance = draw_rotated_distance_overlay(frame, row)
        rotated_overlay_path = chapter_7a_overlay_dir / f"{frame_path.stem}__chapter_7a_overlay.png"
        cv2.imwrite(str(rotated_overlay_path), rotated_overlay_frame)
        chapter_7a_overlay_records.append(
            {
                "row": row,
                "overlay_frame": rotated_overlay_frame,
                "overlay_path": rotated_overlay_path,
                "datum_to_lapbar_distance": datum_to_lapbar_distance,
                "datum_to_bottle_distance": datum_to_bottle_distance,
            }
        )

    if not chapter_7a_overlay_records:
        print("No readable frames were available for overlay review.")
    else:
        # Keep the grid shallow so each rotated frame gets enough screen area.
        chapter_7a_column_count = min(2, len(chapter_7a_overlay_records))
        chapter_7a_row_count = int(np.ceil(len(chapter_7a_overlay_records) / chapter_7a_column_count))
        figure, axes = plt.subplots(
            chapter_7a_row_count,
            chapter_7a_column_count,
            figsize=(8.5 * chapter_7a_column_count, 6.6 * chapter_7a_row_count),
            dpi=180,
        )
        axes = np.atleast_1d(axes).ravel()

        # Plot the rotated overlays with the same title structure as Chapter 7.
        for plot_index, overlay_record in enumerate(chapter_7a_overlay_records):
            row = overlay_record["row"]
            overlay_frame = overlay_record["overlay_frame"]
            datum_to_lapbar_distance = overlay_record["datum_to_lapbar_distance"]
            datum_to_bottle_distance = overlay_record["datum_to_bottle_distance"]
            axes[plot_index].imshow(cv2.cvtColor(overlay_frame, cv2.COLOR_BGR2RGB))
            bottle_title = "not measured" if datum_to_bottle_distance is None else f"{datum_to_bottle_distance:.1f}px"
            axes[plot_index].set_title(
                f"cycle {int(row['cycle_index'])}, phase {int(row['aligned_phase_index'])}, "
                f"lapbar {datum_to_lapbar_distance:.1f}px, bottle {bottle_title}",
                fontsize=13,
            )
            axes[plot_index].axis("off")

        # Hide any unused axes in the grid.
        for axis in axes[len(chapter_7a_overlay_records):]:
            axis.axis("off")

        plt.tight_layout(pad=1.2)
        plt.show()
        print(f"Saved Chapter 7A reference overlays to: {chapter_7a_overlay_dir}")


C:\Users\TimKitchen\AppData\Local\Temp\ipykernel_2844\236826915.py:224: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved Chapter 7A reference overlays to: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\overlays\chapter_7a_reference


## 8. Batch S3 Processing and Save Overlay Frames

Use the same one-video-at-a-time pattern as `motion_detect.py`: list raw clips from S3 inside a fixed timestamp window, download one clip, run the phase-aware measurement flow, append the results, save Chapter 7-style overlay PNGs for valid frames, delete the local clip, and move to the next video.


In [11]:
import re
import boto3

# ------------------------------
# Batch S3 settings
# ------------------------------

# Use the same S3 listing pattern as motion_detect.py.
BATCH_AWS_PROFILE = "DashcamGlbDiageoProdDataContrib-522196013725"
BATCH_S3_BUCKET = "diageo-prod-global-dashcam-mc-nuc-video"
BATCH_S3_PREFIX = "cortexvpu-01a-005-41884872/"
BATCH_VIDEO_EXTENSIONS = [".ts"]

# Set the fixed filename timestamp window to process.
BATCH_START_TIME = "2026-06-19 14:00:00"
BATCH_END_TIME = "2026-06-20 08:15:00"

# Use this to force one named clip during tuning.
BATCH_TEST_VIDEO_FILENAME = None

# Keep this small while tuning Chapter 8. Set to None for a full backfill.
BATCH_MAX_VIDEOS_TO_PROCESS = 2

# Keep Chapter 8 self-contained so it always exports the same aligned Phase 2
# used in the rest of the notebook, even if other globals are changed later.
BATCH_PHASE_TO_REVIEW = int(phase_to_review)
BATCH_PHASES_OF_INTEREST = [BATCH_PHASE_TO_REVIEW]

# Match the Chapter 4 export volume setting.
# None keeps every matching Phase 2 instance.
BATCH_SAMPLES_PER_PHASE_PER_VIDEO = export_samples_per_phase

# Match Chapters 0-7 by not adding extra batch-only NCC confidence thresholds.
# Set these manually only when Chapter 8 needs stricter filtering than the review flow.
BATCH_MIN_KEYFRAME_NCC_SCORE = None
BATCH_MIN_KEYFRAME_NCC_MARGIN = None

# Write all Chapter 8 artefacts into the streamlined batch folder.
batch_output_dir = output_batch_dir
batch_download_dir = batch_output_dir / "downloaded_videos"
batch_exported_frames_dir = batch_output_dir / "frames"
batch_mask_dir = batch_output_dir / "masks"
batch_overlay_dir = batch_output_dir / "overlays"
batch_measurements_dir = batch_output_dir / "measurements"
batch_manifest_dir = batch_output_dir / "manifests"
batch_measurements_output_path = batch_measurements_dir / "batch_position_measurements.csv"
batch_video_summary_path = batch_manifest_dir / "batch_video_summary.csv"
batch_processed_videos_path = batch_manifest_dir / "batch_processed_videos.csv"

for batch_subdir in [
    batch_output_dir,
    batch_download_dir,
    batch_exported_frames_dir,
    batch_mask_dir,
    batch_overlay_dir,
    batch_measurements_dir,
    batch_manifest_dir,
]:
    batch_subdir.mkdir(parents=True, exist_ok=True)

# Reuse the existing reference image and ROI if they are already loaded.
if "reference_image" not in globals():
    reference_image_path = resolve_reference_image_path(reference_dir, reference_image_path)
    reference_image = load_reference_image(reference_image_path)

if "measurement_roi" not in globals():
    if roi_json_path.exists():
        measurement_roi = load_roi(roi_json_path)
    else:
        measurement_roi = None

# Keep Chapter 8 safe when it is run out of order: phase detection uses its own ROI setting.
if "phase_detection_roi" not in globals():
    phase_detection_roi = None
cfg = cfg.replace(phase_roi=phase_detection_roi)

# Load the current Pitwall configs once for the batch run.
lapbar_config = load_pitwall_config(lapbar_config_path)
bottle_config = load_pitwall_config(bottle_config_path)
datum_config = load_pitwall_config(datum_config_path)

# Chapter 8 reuses the Chapter 7 overlay renderer.
if "draw_distance_overlay" not in globals():
    raise RuntimeError("Run Chapter 7 before Chapter 8 so the overlay drawing helper is available.")


def ensure_batch_reference_phase_context() -> tuple[Path, dict, dict]:
    """Reuse or rebuild the same reference-clip phase context used earlier in the notebook."""
    global reference_video_path, reference_phase_data, reference_alignment

    if "reference_video_path" not in globals():
        reference_video_candidates = list_video_paths(reference_dir, max_items=1)
        reference_video_path = reference_video_candidates[0]

    if "reference_phase_data" not in globals():
        reference_phase_data = detect_video_phases(reference_video_path, cfg)

    if "reference_alignment" not in globals():
        reference_alignment = align_video_phases_to_reference_image(
            reference_image=reference_image,
            video_path=reference_video_path,
            cycles=reference_phase_data["phase_result"].cycles,
            cfg=cfg,
            roi=measurement_roi,
        )

    if not reference_alignment["aligned_phase_grid"]:
        raise ValueError("Reference alignment did not produce any aligned phase frames.")

    return reference_video_path, reference_phase_data, reference_alignment


def parse_video_timestamp_from_filename(filename: str, timestamp_pattern: re.Pattern[str]) -> pd.Timestamp | None:
    """Parse the clip timestamp from the raw S3 filename the same way as motion_detect.py."""
    timestamp_match = timestamp_pattern.search(filename)
    if not timestamp_match:
        return None

    return pd.to_datetime(
        timestamp_match.group(1),
        format="%Y-%m-%d_%H-%M-%S_%f",
    )


def list_s3_videos_to_process(
    aws_profile: str | None,
    bucket: str,
    prefix: str,
    video_extensions: list[str],
    start_time_text: str | None,
    end_time_text: str | None,
    test_video_filename: str | None,
    processed_videos_path: Path,
    max_videos_to_process: int | None,
) -> pd.DataFrame:
    """List raw S3 videos inside the requested timestamp window."""
    start_time = pd.to_datetime(start_time_text) if start_time_text else None
    end_time = pd.to_datetime(end_time_text) if end_time_text else None
    timestamp_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d+)")
    test_mode = test_video_filename is not None

    # Build the S3 client the same way as motion_detect.py.
    session = boto3.Session(profile_name=aws_profile) if aws_profile else boto3.Session()
    s3_client = session.client("s3")

    candidate_videos = []
    raw_video_count = 0
    timestamp_match_count = 0

    if test_mode:
        candidate_videos.append(
            {
                "filename": test_video_filename,
                "timestamp": pd.NaT,
                "s3_key": f"{prefix}{test_video_filename}",
            }
        )
        raw_video_count = 1
        timestamp_match_count = 1
    else:
        paginator = s3_client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for s3_object in page.get("Contents", []):
                s3_key = s3_object["Key"]
                filename = Path(s3_key).name

                # Skip generated outputs stored under nested folders in the same prefix.
                if s3_key != f"{prefix}{filename}":
                    continue

                # Only raw video files should enter the queue.
                if Path(s3_key).suffix.lower() not in video_extensions:
                    continue
                raw_video_count += 1

                # Read the clip timestamp from the filename, not from S3 LastModified.
                video_timestamp = parse_video_timestamp_from_filename(filename, timestamp_pattern)
                if video_timestamp is None:
                    continue
                timestamp_match_count += 1

                if start_time is not None and video_timestamp < start_time:
                    continue
                if end_time is not None and video_timestamp > end_time:
                    continue

                candidate_videos.append(
                    {
                        "filename": filename,
                        "timestamp": video_timestamp,
                        "s3_key": s3_key,
                    }
                )

    videos_df = pd.DataFrame(candidate_videos, columns=["filename", "timestamp", "s3_key"])
    if videos_df.empty:
        return videos_df

    # Process clips in timestamp order.
    videos_df = videos_df.sort_values("timestamp", na_position="last").reset_index(drop=True)

    # Skip clips already completed in earlier runs.
    if processed_videos_path.exists():
        processed_videos_df = pd.read_csv(processed_videos_path)
        processed_filenames = set(processed_videos_df["filename"].dropna().astype(str))
    else:
        processed_filenames = set()

    if not test_mode:
        videos_df = videos_df[~videos_df["filename"].isin(processed_filenames)].copy()

    if max_videos_to_process is not None and not test_mode:
        videos_df = videos_df.head(max_videos_to_process).copy()

    print("Batch Chapter 8 settings")
    print(f"  Bucket: s3://{bucket}/{prefix}")
    print(f"  Date range: {start_time} to {end_time}")
    print(f"  Test video: {test_video_filename}")
    print(f"  Max videos this run: {max_videos_to_process}")
    print(f"  Batch phase to review: {BATCH_PHASE_TO_REVIEW}")
    print(f"  Samples per phase per video: {BATCH_SAMPLES_PER_PHASE_PER_VIDEO}")
    print(f"  Min key-frame NCC score: {BATCH_MIN_KEYFRAME_NCC_SCORE}")
    print(f"  Min key-frame NCC margin: {BATCH_MIN_KEYFRAME_NCC_MARGIN}")
    print(f"  Raw videos before date filter: {raw_video_count}")
    print(f"  Videos with parsed timestamps: {timestamp_match_count}")
    print(f"  Videos queued for this run: {len(videos_df)}")
    if not videos_df.empty:
        print(f"  First queued video timestamp: {videos_df['timestamp'].min()}")
        print(f"  Last queued video timestamp: {videos_df['timestamp'].max()}")
        print(f"  First queued filename: {videos_df.iloc[0]['filename']}")
        print(f"  Last queued filename: {videos_df.iloc[-1]['filename']}")

    return videos_df


def ensure_batch_measurement_columns(measurement_df: pd.DataFrame) -> pd.DataFrame:
    """Add any measurement columns that older manifests may not contain yet."""
    updated_df = measurement_df.copy()
    required_defaults = {
        "lapbar_mask_path": "",
        "bottle_mask_path": "",
        "datum_mask_path": "",
        "lapbar_x_from_datum": np.nan,
        "lapbar_y_from_datum": np.nan,
        "bottle_x_from_datum": np.nan,
        "bottle_y_from_datum": np.nan,
        "datum_x_from_frame": np.nan,
        "datum_y_from_frame": np.nan,
        "bottle_roi_count": np.nan,
        "bottle_measurement_valid": False,
        "bottle_middle_roi_full_size": False,
        "bottle_minus_datum_x": np.nan,
        "bottle_minus_datum_y": np.nan,
        "bottle_from_datum_euclidean": np.nan,
        "lapbar_minus_datum_x": np.nan,
        "lapbar_minus_datum_y": np.nan,
        "lapbar_from_datum_euclidean": np.nan,
        "overlay_frame_path": "",
    }

    # Add missing columns one by one so reruns stay backward compatible.
    for column_name, default_value in required_defaults.items():
        if column_name not in updated_df.columns:
            updated_df[column_name] = default_value

    return updated_df


def measure_manifest_rows(manifest_df: pd.DataFrame, bottle_config: dict) -> pd.DataFrame:
    """Apply the shared Chapter 6 measurement logic to one exported-manifest DataFrame."""
    measurement_df = ensure_batch_measurement_columns(manifest_df)
    return populate_manifest_measurements(measurement_df, bottle_config)


def save_overlay_frames_for_measurements(measurement_df: pd.DataFrame, overlay_dir: Path) -> pd.DataFrame:
    """Save one Chapter 7-style overlay PNG per row with datum and lapbar data."""
    updated_df = measurement_df.copy()
    overlay_dir.mkdir(parents=True, exist_ok=True)

    # Match Chapter 7: bottle values may be missing when the middle ROI rule rejects a frame.
    required_overlay_columns = [
        "export_path",
        "bottle_mask_path",
        "lapbar_x_from_datum",
        "lapbar_y_from_datum",
        "datum_x_from_frame",
        "datum_y_from_frame",
        "lapbar_minus_datum_x",
        "lapbar_minus_datum_y",
        "lapbar_from_datum_euclidean",
    ]

    # Keep rows where the lapbar/datum relationship can be drawn.
    valid_rows_df = updated_df.dropna(subset=required_overlay_columns).copy()

    for row_index, row in valid_rows_df.iterrows():
        frame_path = Path(row["export_path"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            print(f"  Skipped unreadable exported frame: {frame_path}")
            continue

        # Reuse the Chapter 7 overlay drawing so the annotations match exactly.
        overlay_frame, _, _ = draw_distance_overlay(frame, row)
        overlay_path = overlay_dir / f"{frame_path.stem}__overlay.png"
        cv2.imwrite(str(overlay_path), overlay_frame)
        updated_df.at[row_index, "overlay_frame_path"] = str(overlay_path)

    return updated_df


def append_dataframe_rows(dataframe: pd.DataFrame, csv_path: Path) -> None:
    """Append rows to a CSV and keep the header aligned when columns change."""
    if dataframe.empty:
        return

    # Create the CSV directly when this is the first write.
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        dataframe.to_csv(csv_path, index=False)
        return

    # Existing batch files may have an older column set, so widen before writing.
    existing_df = pd.read_csv(csv_path)
    all_columns = list(existing_df.columns)
    for column_name in dataframe.columns:
        if column_name not in all_columns:
            all_columns.append(column_name)

    existing_df = existing_df.reindex(columns=all_columns)
    new_rows_df = dataframe.reindex(columns=all_columns)
    combined_df = pd.concat([existing_df, new_rows_df], ignore_index=True)
    combined_df.to_csv(csv_path, index=False)


def append_processed_video(filename: str, processed_videos_path: Path) -> None:
    """Mark one video as complete after its outputs have been written."""
    processed_row_df = pd.DataFrame([{"filename": filename}])
    if processed_videos_path.exists():
        existing_df = pd.read_csv(processed_videos_path)
        processed_df = pd.concat([existing_df, processed_row_df], ignore_index=True).drop_duplicates()
    else:
        processed_df = processed_row_df
    processed_df.to_csv(processed_videos_path, index=False)


def should_skip_video_for_phase_awareness(error: Exception) -> bool:
    """Return True when a clip has no usable phase-awareness output and should be skipped."""
    error_text = str(error)
    skip_markers = [
        "No cycles were found",
        "Could not build an ROI phase grid",
        "Could not build a key-frame scoring grid",
        "Could not build a full-frame phase grid",
        "No key-frame records were created",
        "No high-confidence key-frame records",
    ]

    for skip_marker in skip_markers:
        if skip_marker in error_text:
            return True

    return False


def get_reference_phase_frame(reference_alignment: dict, phase_index: int) -> np.ndarray:
    """Return the aligned reference-clip frame that defines the target phase."""
    # Use the first reference cycle that contains the requested phase.
    for cycle_frames in reference_alignment["aligned_phase_grid"]:
        if len(cycle_frames) > phase_index:
            return cycle_frames[phase_index]

    raise ValueError(f"Reference alignment did not contain aligned phase {phase_index}.")


def is_keyframe_ncc_match_confident(
    ncc_score: float,
    ncc_margin: float,
    min_ncc_score: float | None,
    min_ncc_margin: float | None,
) -> bool:
    """Return True when the key-frame NCC match is strong enough to keep."""
    # Reject weak absolute matches first.
    if min_ncc_score is not None and ncc_score < min_ncc_score:
        return False

    # Reject ambiguous matches where the best phase barely beats second best.
    if min_ncc_margin is not None:
        if pd.isna(ncc_margin):
            return False
        if ncc_margin < min_ncc_margin:
            return False

    return True


def build_full_frame_keyframe_records_by_ncc(
    video_path: Path,
    cycles: list[tuple[int, int]],
    full_frames: list[np.ndarray],
    reference_phase_frame: np.ndarray,
    target_aligned_phase_index: int,
    cfg: PhaseDetectionConfig,
    roi,
    min_ncc_score: float | None,
    min_ncc_margin: float | None,
) -> tuple[list[dict], dict]:
    """Select high-confidence full-frame key frames by NCC to the reference phase frame."""
    if not cycles:
        raise ValueError(f"No detected cycles were available for key-frame selection: {video_path.name}")

    if len(full_frames) == 0:
        raise ValueError(f"No decoded frames were available for key-frame selection: {video_path.name}")

    # The reference phase frame is already in the correct scope, so do not crop it again here.
    reference_ncc_image = prepare_ncc_image(reference_phase_frame, None, phase_ncc_resize_dim)

    keyframe_records = []
    rejected_low_score_count = 0
    rejected_low_margin_count = 0
    rejected_scores = []
    rejected_margins = []
    candidate_counts = []
    usable_cycle_count = 0

    for cycle_index, raw_cycle in enumerate(cycles):
        raw_cycle_start, raw_cycle_end = raw_cycle

        # Clamp the detected cycle to the frames that were actually decoded.
        cycle_start = max(0, int(raw_cycle_start))
        cycle_end = min(len(full_frames) - 1, int(raw_cycle_end))

        # Skip broken cycles rather than creating an invalid candidate list.
        if cycle_end < cycle_start:
            continue

        # Search every real frame in the detected cycle, not only the resampled phase points.
        candidate_frame_indices = list(range(cycle_start, cycle_end + 1))
        candidate_counts.append(len(candidate_frame_indices))
        usable_cycle_count += 1

        # Score each candidate frame against the reference Phase 2 image.
        candidate_scores = []
        for frame_index in candidate_frame_indices:
            full_frame = full_frames[frame_index]
            if roi is None:
                scoring_frame = full_frame
            else:
                scoring_frame = crop_to_roi(full_frame, roi)

            candidate_image = prepare_ncc_image(scoring_frame, None, phase_ncc_resize_dim)
            candidate_scores.append(float(compute_ncc_opencv(candidate_image, reference_ncc_image)))

        # Keep the real frame that best matches the target reference phase.
        best_candidate_position = int(np.argmax(candidate_scores))
        best_frame_index = candidate_frame_indices[best_candidate_position]
        sorted_candidate_scores = sorted(candidate_scores, reverse=True)
        best_score = float(sorted_candidate_scores[0])
        second_best_score = float(sorted_candidate_scores[1]) if len(sorted_candidate_scores) > 1 else np.nan
        score_margin = best_score - second_best_score if not pd.isna(second_best_score) else np.nan

        # Preserve the old phase fields by estimating where the chosen frame sits in the cycle.
        cycle_frame_count = max(cycle_end - cycle_start + 1, 1)
        relative_cycle_position = (best_frame_index - cycle_start) / max(cycle_frame_count - 1, 1)
        best_original_phase_index = int(round(relative_cycle_position * (cfg.num_phases - 1)))
        inferred_phase_zero_source = (best_original_phase_index - target_aligned_phase_index) % cfg.num_phases

        # Drop weak or ambiguous matches before any export, mask generation, or measurement.
        is_confident_match = is_keyframe_ncc_match_confident(
            ncc_score=best_score,
            ncc_margin=score_margin,
            min_ncc_score=min_ncc_score,
            min_ncc_margin=min_ncc_margin,
        )
        if not is_confident_match:
            rejected_scores.append(best_score)
            rejected_margins.append(score_margin)
            if min_ncc_score is not None and best_score < min_ncc_score:
                rejected_low_score_count += 1
            elif min_ncc_margin is not None and (pd.isna(score_margin) or score_margin < min_ncc_margin):
                rejected_low_margin_count += 1
            continue

        keyframe_records.append(
            {
                "video_name": video_path.name,
                "cycle_index": cycle_index,
                "aligned_phase_index": target_aligned_phase_index,
                "original_phase_index": best_original_phase_index,
                "phase_zero_source_phase_index": inferred_phase_zero_source,
                "phase_zero_match_score": best_score,
                "keyframe_ncc_score": best_score,
                "keyframe_ncc_margin": score_margin,
                "keyframe_frame_index": best_frame_index,
                "keyframe_cycle_start": cycle_start,
                "keyframe_cycle_end": cycle_end,
                "keyframe_candidate_count": len(candidate_frame_indices),
                "phase_selection_method": "direct_reference_frame_ncc_within_detected_cycle",
                "frame": full_frames[best_frame_index],
            }
        )

    keyframe_diagnostics = {
        "candidate_cycles": usable_cycle_count,
        "accepted_keyframes": len(keyframe_records),
        "rejected_keyframes": usable_cycle_count - len(keyframe_records),
        "rejected_low_score_count": rejected_low_score_count,
        "rejected_low_margin_count": rejected_low_margin_count,
        "rejected_ncc_score_median": float(np.nanmedian(rejected_scores)) if rejected_scores else np.nan,
        "rejected_ncc_margin_median": float(np.nanmedian(rejected_margins)) if rejected_margins else np.nan,
        "min_keyframe_ncc_score": min_ncc_score,
        "min_keyframe_ncc_margin": min_ncc_margin,
        "keyframe_search_scope": "all_frames_within_detected_cycle",
        "keyframe_candidate_count_median": float(np.nanmedian(candidate_counts)) if candidate_counts else np.nan,
    }

    if not keyframe_records:
        raise ValueError(
            f"No high-confidence key-frame records were created for: {video_path.name}. "
            f"Best rejected NCC median: {keyframe_diagnostics['rejected_ncc_score_median']}"
        )

    return keyframe_records, keyframe_diagnostics


def run_batch_measurement_for_video(
    local_video_path: Path,
    video_filename: str,
    video_timestamp,
    reference_phase_frame: np.ndarray,
    measurement_roi,
    lapbar_config: dict,
    bottle_config: dict,
    datum_config: dict,
    phases_of_interest: list[int],
    samples_per_phase_per_video: int | None,
) -> tuple[pd.DataFrame, dict]:
    """Run the phase-aware export, mask, measurement, and overlay flow for one video."""
    # Write all videos into the same batch folders. The filename already includes the video stem.
    video_export_dir = batch_exported_frames_dir
    video_mask_dir = batch_mask_dir
    video_overlay_dir = batch_overlay_dir

    video_export_dir.mkdir(parents=True, exist_ok=True)
    video_mask_dir.mkdir(parents=True, exist_ok=True)
    video_overlay_dir.mkdir(parents=True, exist_ok=True)

    detected_cycles = np.nan

    try:
        # Detect cycles first, then select the target phase by direct NCC to the reference Phase 2 frame.
        video_phase_data = detect_video_phases(local_video_path, cfg)
        detected_cycles = len(video_phase_data["phase_result"].cycles)
        keyframe_records, keyframe_diagnostics = build_full_frame_keyframe_records_by_ncc(
            video_path=local_video_path,
            cycles=video_phase_data["phase_result"].cycles,
            full_frames=video_phase_data["frames"],
            reference_phase_frame=reference_phase_frame,
            target_aligned_phase_index=BATCH_PHASE_TO_REVIEW,
            cfg=cfg,
            roi=measurement_roi,
            min_ncc_score=BATCH_MIN_KEYFRAME_NCC_SCORE,
            min_ncc_margin=BATCH_MIN_KEYFRAME_NCC_MARGIN,
        )
    except Exception as exc:
        if not should_skip_video_for_phase_awareness(exc):
            raise

        # Distinguish failed phase detection from low-confidence key-frame matches.
        error_text = str(exc)
        skip_status = "skipped_low_keyframe_ncc" if "No high-confidence key-frame records" in error_text else "skipped_no_phase_awareness"

        summary_row = {
            "video_name": video_filename,
            "video_timestamp": None if pd.isna(video_timestamp) else pd.Timestamp(video_timestamp).isoformat(),
            "detected_cycles": detected_cycles,
            "aligned_phase_zero_source": np.nan,
            "keyframe_original_phase_mode": np.nan,
            "keyframe_ncc_score_median": np.nan,
            "keyframe_ncc_margin_median": np.nan,
            "candidate_cycles": detected_cycles,
            "accepted_keyframes": 0,
            "rejected_keyframes": detected_cycles,
            "rejected_low_score_count": np.nan,
            "rejected_low_margin_count": np.nan,
            "rejected_ncc_score_median": np.nan,
            "rejected_ncc_margin_median": np.nan,
            "min_keyframe_ncc_score": BATCH_MIN_KEYFRAME_NCC_SCORE,
            "min_keyframe_ncc_margin": BATCH_MIN_KEYFRAME_NCC_MARGIN,
            "exported_frames": 0,
            "valid_measurements": 0,
            "overlay_frames_saved": 0,
            "export_frame_scope": "not_exported",
            "phase_selection_method": "direct_reference_frame_ncc_within_detected_cycle",
            "status": skip_status,
            "error": str(exc),
        }
        return pd.DataFrame(), summary_row

    # Export the requested reference-matched full-frame key frames for this one video.
    exported_phase_frames_df = export_phase_frames(
        frame_records=keyframe_records,
        export_dir=video_export_dir,
        phases_of_interest=phases_of_interest,
        samples_per_phase=samples_per_phase_per_video,
    )

    selected_original_phases = [record["original_phase_index"] for record in keyframe_records]
    selected_scores = [record["keyframe_ncc_score"] for record in keyframe_records]
    selected_margins = [record["keyframe_ncc_margin"] for record in keyframe_records]
    keyframe_original_phase_mode = int(pd.Series(selected_original_phases).mode().iloc[0])
    inferred_phase_zero_source = (keyframe_original_phase_mode - BATCH_PHASE_TO_REVIEW) % cfg.num_phases

    if exported_phase_frames_df.empty:
        summary_row = {
            "video_name": video_filename,
            "video_timestamp": None if pd.isna(video_timestamp) else pd.Timestamp(video_timestamp).isoformat(),
            "detected_cycles": len(video_phase_data["phase_result"].cycles),
            "aligned_phase_zero_source": inferred_phase_zero_source,
            "keyframe_original_phase_mode": keyframe_original_phase_mode,
            "keyframe_ncc_score_median": float(np.nanmedian(selected_scores)),
            "keyframe_ncc_margin_median": float(np.nanmedian(selected_margins)),
            **keyframe_diagnostics,
            "exported_frames": 0,
            "valid_measurements": 0,
            "overlay_frames_saved": 0,
            "export_frame_scope": "full_frame",
            "phase_selection_method": "direct_reference_frame_ncc_within_detected_cycle",
            "status": "no_exported_frames",
        }
        return exported_phase_frames_df, summary_row

    # Build a fresh manifest, generate masks, and then measure each frame.
    phase_frame_manifest_df = build_phase_frame_manifest(exported_phase_frames_df)
    phase_frame_manifest_df = write_generated_masks_for_manifest(
        manifest_df=phase_frame_manifest_df,
        lapbar_config=lapbar_config,
        bottle_config=bottle_config,
        datum_config=datum_config,
        output_dir=video_mask_dir,
        phase_index=BATCH_PHASE_TO_REVIEW,
    )

    measurement_df = measure_manifest_rows(phase_frame_manifest_df, bottle_config)
    measurement_df["source_video_name"] = video_filename
    measurement_df["source_video_timestamp"] = None if pd.isna(video_timestamp) else pd.Timestamp(video_timestamp).isoformat()
    measurement_df = save_overlay_frames_for_measurements(measurement_df, video_overlay_dir)

    valid_measurements = measurement_df[
        measurement_df["bottle_measurement_valid"].astype(bool)
        & measurement_df["bottle_middle_roi_full_size"].astype(bool)
    ].copy()
    overlay_frames_saved = int(measurement_df["overlay_frame_path"].astype(str).str.len().gt(0).sum())

    summary_row = {
        "video_name": video_filename,
        "video_timestamp": None if pd.isna(video_timestamp) else pd.Timestamp(video_timestamp).isoformat(),
        "detected_cycles": len(video_phase_data["phase_result"].cycles),
        "aligned_phase_zero_source": inferred_phase_zero_source,
        "keyframe_original_phase_mode": keyframe_original_phase_mode,
        "keyframe_ncc_score_median": float(np.nanmedian(selected_scores)),
        "keyframe_ncc_margin_median": float(np.nanmedian(selected_margins)),
        **keyframe_diagnostics,
        "exported_frames": len(exported_phase_frames_df),
        "valid_measurements": len(valid_measurements),
        "overlay_frames_saved": overlay_frames_saved,
        "export_frame_scope": "full_frame",
        "phase_selection_method": "direct_reference_frame_ncc_within_detected_cycle",
        "status": "ok",
    }

    return measurement_df, summary_row


batch_reference_video_path, batch_reference_phase_data, batch_reference_alignment = ensure_batch_reference_phase_context()
batch_reference_phase_frame = get_reference_phase_frame(batch_reference_alignment, BATCH_PHASE_TO_REVIEW)
print(f"Chapter 8 reference clip: {batch_reference_video_path.name}")
print(f"Chapter 8 reference detected cycles: {len(batch_reference_phase_data['phase_result'].cycles)}")
print(f"Chapter 8 reference aligned phase 0 source: {batch_reference_alignment['best_original_phase_index']}")
print(f"Chapter 8 NCC key-frame reference: aligned reference clip phase {BATCH_PHASE_TO_REVIEW}")
print(f"Chapter 8 phase review target: {BATCH_PHASE_TO_REVIEW}")

batch_videos_df = list_s3_videos_to_process(
    aws_profile=BATCH_AWS_PROFILE,
    bucket=BATCH_S3_BUCKET,
    prefix=BATCH_S3_PREFIX,
    video_extensions=BATCH_VIDEO_EXTENSIONS,
    start_time_text=BATCH_START_TIME,
    end_time_text=BATCH_END_TIME,
    test_video_filename=BATCH_TEST_VIDEO_FILENAME,
    processed_videos_path=batch_processed_videos_path,
    max_videos_to_process=BATCH_MAX_VIDEOS_TO_PROCESS,
)

if batch_videos_df.empty:
    print("No S3 videos matched the Chapter 8 settings.")
else:
    # Create the S3 client once, then process each video independently.
    batch_session = boto3.Session(profile_name=BATCH_AWS_PROFILE) if BATCH_AWS_PROFILE else boto3.Session()
    s3_client = batch_session.client("s3")
    test_mode = BATCH_TEST_VIDEO_FILENAME is not None

    for video_index, video_row in enumerate(batch_videos_df.itertuples(index=False), start=1):
        filename = video_row.filename
        video_timestamp = video_row.timestamp
        s3_key = video_row.s3_key
        local_video_path = batch_download_dir / filename

        print("----------------------------------------------------")
        print(f"Processing Chapter 8 video {video_index}/{len(batch_videos_df)}")
        print(f"  Filename: {filename}")
        print(f"  S3 key: {s3_key}")
        print(f"  Downloading to: {local_video_path}")

        s3_client.download_file(BATCH_S3_BUCKET, s3_key, str(local_video_path))

        summary_row = {
            "video_name": filename,
            "video_timestamp": None if pd.isna(video_timestamp) else pd.Timestamp(video_timestamp).isoformat(),
            "detected_cycles": np.nan,
            "aligned_phase_zero_source": np.nan,
            "keyframe_original_phase_mode": np.nan,
            "keyframe_ncc_score_median": np.nan,
            "keyframe_ncc_margin_median": np.nan,
            "candidate_cycles": np.nan,
            "accepted_keyframes": 0,
            "rejected_keyframes": np.nan,
            "rejected_low_score_count": np.nan,
            "rejected_low_margin_count": np.nan,
            "rejected_ncc_score_median": np.nan,
            "rejected_ncc_margin_median": np.nan,
            "min_keyframe_ncc_score": BATCH_MIN_KEYFRAME_NCC_SCORE,
            "min_keyframe_ncc_margin": BATCH_MIN_KEYFRAME_NCC_MARGIN,
            "exported_frames": 0,
            "valid_measurements": 0,
            "overlay_frames_saved": 0,
            "export_frame_scope": "not_exported",
            "phase_selection_method": "direct_reference_frame_ncc_within_detected_cycle",
            "status": "failed",
            "error": None,
        }

        try:
            measurement_df, per_video_summary_row = run_batch_measurement_for_video(
                local_video_path=local_video_path,
                video_filename=filename,
                video_timestamp=video_timestamp,
                reference_phase_frame=batch_reference_phase_frame,
                measurement_roi=measurement_roi,
                lapbar_config=lapbar_config,
                bottle_config=bottle_config,
                datum_config=datum_config,
                phases_of_interest=BATCH_PHASES_OF_INTEREST,
                samples_per_phase_per_video=BATCH_SAMPLES_PER_PHASE_PER_VIDEO,
            )

            # Append the per-frame results before marking the video complete.
            append_dataframe_rows(measurement_df, batch_measurements_output_path)
            summary_row.update(per_video_summary_row)
            if "error" not in per_video_summary_row:
                summary_row["error"] = None

            if not test_mode:
                append_processed_video(filename, batch_processed_videos_path)
                print(f"  Marked as processed in: {batch_processed_videos_path}")
            else:
                print("  Test mode: processed-video tracking not updated")

            if summary_row["status"] == "skipped_no_phase_awareness":
                print(f"  Skipped: {summary_row['error']}")
            elif summary_row["status"] == "skipped_low_keyframe_ncc":
                print("  Skipped: no high-confidence key frames were exported")
            elif summary_row.get("accepted_keyframes", 0) == 0:
                print("  Skipped: no high-confidence key frames were exported")
            else:
                print(f"  Appended frame measurements to: {batch_measurements_output_path}")
                print(f"  Saved overlay frames in: {batch_overlay_dir}")

        except Exception as exc:
            summary_row["error"] = str(exc)
            print(f"  Failed: {exc}")

        finally:
            # Always append the per-video summary, even on failure.
            append_dataframe_rows(pd.DataFrame([summary_row]), batch_video_summary_path)

            # Always remove the downloaded source clip after this video is handled.
            if local_video_path.exists():
                local_video_path.unlink()
                print(f"  Deleted local video: {local_video_path}")

        print("----------------------------------------------------")

    if batch_video_summary_path.exists():
        display(pd.read_csv(batch_video_summary_path).tail(len(batch_videos_df)))

print(f"Batch measurements CSV: {batch_measurements_output_path}")
print(f"Batch video summary CSV: {batch_video_summary_path}")
print(f"Batch processed videos CSV: {batch_processed_videos_path}")
print(f"Batch overlay frame folder: {batch_overlay_dir}")


Chapter 8 reference clip: cortexvpu-01a-005-41884872_2026-06-19_17-35-41_933333_clip.ts
Chapter 8 reference detected cycles: 46
Chapter 8 reference aligned phase 0 source: 8
Chapter 8 NCC key-frame reference: aligned reference clip phase 3
Chapter 8 phase review target: 3
Batch Chapter 8 settings
  Bucket: s3://diageo-prod-global-dashcam-mc-nuc-video/cortexvpu-01a-005-41884872/
  Date range: 2026-06-19 14:00:00 to 2026-06-20 08:15:00
  Test video: None
  Max videos this run: 2
  Batch phase to review: 3
  Samples per phase per video: None
  Min key-frame NCC score: None
  Min key-frame NCC margin: None
  Raw videos before date filter: 23295
  Videos with parsed timestamps: 23295
  Videos queued for this run: 2
  First queued video timestamp: 2026-06-19 14:01:13.904907
  Last queued video timestamp: 2026-06-19 14:03:09.910258
  First queued filename: cortexvpu-01a-005-41884872_2026-06-19_14-01-13_904907.ts
  Last queued filename: cortexvpu-01a-005-41884872_2026-06-19_14-03-09_910258.ts


,video_name,video_timestamp,detected_cycles,aligned_phase_zero_source,keyframe_original_phase_mode,keyframe_ncc_score_median,keyframe_ncc_margin_median,candidate_cycles,accepted_keyframes,rejected_keyframes,...,rejected_ncc_margin_median,min_keyframe_ncc_score,min_keyframe_ncc_margin,exported_frames,valid_measurements,overlay_frames_saved,export_frame_scope,phase_selection_method,status,error
0,cortexvpu-01a-005-41884872_2026-06-19_14-01-13...,2026-06-19T14:01:13.904907,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,...,NaN,NaN,NaN,0,0,0,not_exported,direct_reference_frame_ncc_within_detected_cycle,failed,<method 'read' of 'cv2.VideoCapture' objects> ...
1,cortexvpu-01a-005-41884872_2026-06-19_14-03-09...,2026-06-19T14:03:09.910258,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,...,NaN,NaN,NaN,0,0,0,not_exported,direct_reference_frame_ncc_within_detected_cycle,failed,OpenCV(4.13.0) D:\a\opencv-python\opencv-pytho...


Batch measurements CSV: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\batch\measurements\batch_position_measurements.csv
Batch video summary CSV: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\batch\manifests\batch_video_summary.csv
Batch processed videos CSV: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\batch\manifests\batch_processed_videos.csv
Batch overlay frame folder: c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\measurements\phase_aware_position_output\batch\overlays
